**UNIVERSIDAD PONTIFICIA BOLIVARIANA – UPB**  
**Maestría en Ciencia de Datos**

**Asignatura:** Minería de Datos  
**Docente:** Ana Oviedo  
**Elaborado por:** Maryory Patricia Villamizar León  

**Proyecto final:** SNIES Analytics 2020–2025  
**Título:** Segmentación de la demanda académica de educación superior mediante técnicas de minería de datos y algoritmos de agrupamiento K-means.

**Descripción:** Desarrollo de una solución analítica e interactiva para caracterizar y segmentar la demanda de programas académicos de educación superior en Colombia, a partir de información del Sistema Nacional de Información de la Educación Superior (SNIES), correspondiente al período 2020–2025. El proyecto aplica la metodología CRISP-DM, técnicas de preparación de datos y aprendizaje no supervisado mediante K-means. Los resultados se presentan en una aplicación desarrollada con Streamlit, que permite consultar ofertas académicas, visualizar indicadores e interpretar los perfiles identificados.

**Año:** 2026


# CLUSTERING DE PROGRAMAS DE INGENIERÍA — SNIES 2020–2025

**Técnica:** aprendizaje no supervisado mediante K-means.

**Unidad de análisis:** oferta académica de programas de ingeniería.

**Escenarios de estudio:**
- A. Demanda reciente (2025).
- B. Evolución histórica (2020–2025).
- C. Demanda integral (2020–2025 y situación de 2025).

**Etapas metodológicas:**
1. Preparación y exploración de datos.
2. Aprendizaje del modelo: método del codo, silueta y K-means.
3. Evaluación: inercia y coeficiente de silueta.
4. Perfilamiento: centroides, caracterización e identificación de ofertas.


In [ ]:
#Importamos librerías básicas
import pandas as pd # manipulacion dataframes
import numpy as np  # matrices y vectores
import matplotlib.pyplot as plt #gráfica




# 1. Preparación de datos

Se parte de la base SNIES 2020–2025 previamente integrada,
estandarizada y auditada.

La preparación comprende:

- Identificación de ofertas académicas mediante cuatro códigos.
- Definición de criterios de elegibilidad para los escenarios A, B y C.
- Conservación de la diferenciación por semestre y sexo.
- Construcción de indicadores de demanda y tendencias históricas.
- Análisis exploratorio de variables cuantitativas y categóricas.
- Revisión de tipos de datos.
- Normalización de variables numéricas mediante MinMaxScaler.
- Codificación de variables categóricas mediante dummies.

**Criterio de calidad:** no se imputan valores faltantes ni se
sustituyen por cero. Las matrices analíticas se construyen
con información completa para las características requeridas.

No se aplica balanceo de clases por tratarse de aprendizaje
no supervisado, sin variable objetivo.











In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import pandas as pd
from scipy.io import arff
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta del archivo en Google Drive
ruta = "/content/drive/MyDrive/MaestriaUPB_Mineria_Datos/P_final/Educacion_Superior_UNICO_Estudiantes_Personal_2020_2025.arff"

# Cargar los datos en formato ARFF
datos, metadata = arff.loadarff(ruta)

# Convertir los datos a DataFrame
df = pd.DataFrame(datos)

# Convertir variables categóricas de bytes a texto
for columna in df.select_dtypes(include=['object']).columns:
    df[columna] = df[columna].apply(
        lambda x: x.decode('utf-8') if isinstance(x, bytes) else x
    )

# Mostrar los primeros 5 registros
df.head()


In [ ]:

# ============================================================
# PROYECTO SNIES 2020-2025
# ETAPA 1. LIMPIEZA Y ESTANDARIZACIÓN DEL ARCHIVO WEKA
# CONECTADA CON LA CELDA 1
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# ============================================================
# 1. PARTIR EXCLUSIVAMENTE DEL ARFF ORIGINAL
# ============================================================

# Se utiliza df, creado en la primera celda.
assert "df" in globals(), (
    "Primero debes ejecutar la celda 1 de carga del ARFF."
)

assert df.shape == (3896, 58), (
    f"Dimensiones inesperadas de la fuente: {df.shape}. "
    "Vuelve a ejecutar la celda 1 para cargar el ARFF original."
)

# Conservar una copia independiente de la fuente original
df_weka_original = df.copy(deep=True)

# Crear otra copia para estandarizar
df_estandarizada = df_weka_original.copy(deep=True)

print("Base original cargada:", df_weka_original.shape)

# ============================================================
# 2. IDENTIFICADORES DE OFERTA ACADÉMICA
# ============================================================

claves_oferta = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

assert all(
    columna in df_estandarizada.columns
    for columna in claves_oferta
), "Faltan identificadores de oferta académica."

# ============================================================
# 3. FUNCIÓN DE NORMALIZACIÓN TEXTUAL
# ============================================================

def estandarizar_texto(serie):

    return (
        serie.astype('string')
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
        .str.replace(r'\s+', ' ', regex=True)
    )

# ============================================================
# 4. ESTANDARIZAR NOMBRES
# SIN MODIFICAR LOS CAMPOS ORIGINALES
# ============================================================

df_estandarizada['NOMBRE_ESTANDARIZADO'] = (
    estandarizar_texto(
        df_estandarizada['PROGRAMA ACADEMICO']
    )
)

df_estandarizada['MUNICIPIO_ESTANDARIZADO'] = (
    estandarizar_texto(
        df_estandarizada[
            'MUNICIPIO DE OFERTA DEL PROGRAMA'
        ]
    )
)

# ============================================================
# 5. NORMALIZAR IDENTIFICADORES DE OFERTA
# ============================================================

for columna in claves_oferta:

    df_estandarizada[columna] = (
        df_estandarizada[columna]
        .astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )

# ============================================================
# 6. VALIDAR INTEGRIDAD DE LOS DATOS
# ============================================================

assert df_estandarizada.shape == (3896, 60), (
    f"Dimensiones inesperadas: {df_estandarizada.shape}"
)

# Confirmar que no cambió la distribución por sexo
assert (
    df_estandarizada['SEXO']
    .value_counts(dropna=False)
    .equals(
        df_weka_original['SEXO']
        .value_counts(dropna=False)
    )
), "Cambió la distribución de SEXO."

# Confirmar que las variables cuantitativas
# conservan exactamente sus valores originales

variables_cuantitativas = [
    'INSCRITOS',
    'ADMITIDOS',
    'MATRICULADOS PRIMER CURSO',
    'MATRICULADOS',
    'GRADUADOS'
]

assert df_estandarizada[variables_cuantitativas].equals(
    df_weka_original[variables_cuantitativas]
), "Se modificaron los valores cuantitativos."

# Verificar que la fuente original no fue alterada
assert df_weka_original.shape == (3896, 58)

# ============================================================
# 7. GUARDAR CSV Y EXCEL
# ============================================================

carpeta = Path(
    '/content/drive/MyDrive/'
    'MaestriaUPB_Mineria_Datos/P_final'
)

carpeta.mkdir(
    parents=True,
    exist_ok=True
)

ruta_csv = (
    carpeta / 'SNIES_2020_2025_BASE_ESTANDARIZADA.csv'
)

ruta_excel = (
    carpeta / 'SNIES_2020_2025_BASE_ESTANDARIZADA.xlsx'
)

df_estandarizada.to_csv(
    ruta_csv,
    index=False,
    encoding='utf-8-sig'
)

df_estandarizada.to_excel(
    ruta_excel,
    index=False,
    engine='openpyxl'
)

# ============================================================
# 8. VERIFICAR ARCHIVOS GUARDADOS
# ============================================================

verificacion = pd.read_csv(
    ruta_csv,
    encoding='utf-8-sig',
    dtype={c: str for c in claves_oferta},
    low_memory=False
)

assert verificacion.shape == (3896, 60), (
    "El archivo CSV guardado no tiene "
    "las dimensiones esperadas."
)

# ============================================================
# 9. ESTABLECER BASE PARA LA AUDITORÍA
# ============================================================

# Las celdas siguientes continuarán trabajando con df.
# La fuente original queda preservada en df_weka_original.

df = df_estandarizada.copy(deep=True)

# ============================================================
# 10. RESULTADOS DE LA ESTANDARIZACIÓN
# ============================================================

print("\n" + "=" * 60)
print("ESTANDARIZACIÓN DE LA BASE ORIGINAL COMPLETADA")
print("=" * 60)

print("Base original:", df_weka_original.shape)
print("Base estandarizada:", df.shape)

print("\nDISTRIBUCIÓN POR SEXO")
display(
    df['SEXO'].value_counts(dropna=False)
)

print("\nOFERTAS ACADÉMICAS")
print(
    df[claves_oferta]
    .drop_duplicates()
    .shape[0]
)

print("\nEJEMPLOS DE ESTANDARIZACIÓN")

display(
    df[
        [
            'PROGRAMA ACADEMICO',
            'NOMBRE_ESTANDARIZADO',
            'MUNICIPIO DE OFERTA DEL PROGRAMA',
            'MUNICIPIO_ESTANDARIZADO'
        ]
    ]
    .drop_duplicates()
    .head(15)
)

print("\nARCHIVOS GENERADOS")
print("CSV:", ruta_csv)
print("Excel:", ruta_excel)

print("\nBASE PREPARADA PARA LA SIGUIENTE ETAPA.")


#VERIFICACION


In [ ]:

# ============================================================
# SNIES 2020-2025 | AUDITORÍA INTEGRAL DE DATOS
# Maryory Patricia Villamizar León
# Base estandarizada: 3896 registros x 60 variables
# ============================================================

import pandas as pd
from IPython.display import display

# ============================================================
# 1. VARIABLES Y CLAVES
# ============================================================

claves_oferta = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

claves_periodo = claves_oferta + ['ANO', 'SEMESTRE']
claves_registro = claves_periodo + ['SEXO']

variables_cluster = [
    'INSCRITOS',
    'ADMITIDOS',
    'MATRICULADOS PRIMER CURSO',
    'MATRICULADOS',
    'GRADUADOS'
]

def mostrar(titulo, tabla):
    print(f"\n{'=' * 60}\n{titulo}\n{'=' * 60}")
    display(tabla)

# Verificar que se audita la base ESTANDARIZADA
assert df.shape == (3896, 60), (
    f'Se esperaba la base estandarizada (3896, 60), '
    f'pero df tiene dimensiones {df.shape}'
)

# ============================================================
# 2. ESTRUCTURA Y DISTRIBUCIÓN
# ============================================================

n_ofertas = df[claves_oferta].drop_duplicates().shape[0]
duplicados = df.duplicated(claves_registro).sum()

print(
    'Base:', df.shape,
    '| Ofertas:', n_ofertas,
    '| Duplicados:', duplicados
)

for titulo, filas, columnas in [
    ('REGISTROS POR AÑO Y SEMESTRE', 'ANO', 'SEMESTRE'),
    ('REGISTROS POR SEXO Y SEMESTRE', 'SEMESTRE', 'SEXO'),
    ('REGISTROS POR AÑO Y SEXO', 'ANO', 'SEXO')
]:
    mostrar(titulo, pd.crosstab(df[filas], df[columnas]))

# ============================================================
# 3. VALORES FALTANTES POR AÑO Y SEMESTRE
# ============================================================

for agrupacion, nombre in [
    ('ANO', 'AÑO'),
    (['ANO', 'SEMESTRE'], 'AÑO Y SEMESTRE')
]:
    grupo = df.groupby(agrupacion)[variables_cluster]

    faltantes = grupo.agg(lambda x: x.isna().sum())
    faltantes['TOTAL FALTANTES'] = faltantes.sum(axis=1)

    mostrar(
        f'FALTANTES POR {nombre}',
        faltantes
    )

    mostrar(
        f'% FALTANTES POR {nombre}',
        (
            100 -
            grupo.count().div(
                grupo.size(), axis=0
            ) * 100
        ).round(2)
    )

# ============================================================
# 4. COBERTURA POR SEXO
# ============================================================

# Identificar las categorías de sexo presentes
# en cada combinación oferta-año-semestre.

cobertura_sexo = (
    df.groupby(claves_periodo, dropna=False)['SEXO']
    .agg(
        NUM_CATEGORIAS='nunique',
        CATEGORIAS=lambda x: frozenset(x.dropna())
    )
    .reset_index()
)

# Verificar específicamente presencia de Masculino y Femenino.
# También se contabilizan los períodos con categorías adicionales.

cobertura_sexo['MASCULINO_Y_FEMENINO'] = (
    cobertura_sexo['CATEGORIAS'].apply(
        lambda s: {'Masculino', 'Femenino'}.issubset(s)
    )
)

mostrar(
    'CATEGORÍAS DE SEXO POR OFERTA-PERÍODO',
    cobertura_sexo['NUM_CATEGORIAS']
    .value_counts()
    .sort_index()
    .to_frame('OFERTAS_PERIODOS')
)

# Casos donde se reporta una única categoría de sexo
casos_un_sexo = cobertura_sexo.loc[
    cobertura_sexo['NUM_CATEGORIAS'] == 1
].copy()

casos_un_sexo['SEXO_REPORTADO'] = (
    casos_un_sexo['CATEGORIAS'].apply(
        lambda categorias: next(iter(categorias))
    )
)

mostrar(
    'CASOS CON UNA SOLA CATEGORÍA DE SEXO POR AÑO',
    pd.crosstab(
        casos_un_sexo['ANO'],
        casos_un_sexo['SEXO_REPORTADO']
    )
)

mostrar(
    'CASOS CON UNA SOLA CATEGORÍA DE SEXO POR SEMESTRE',
    pd.crosstab(
        casos_un_sexo['SEMESTRE'],
        casos_un_sexo['SEXO_REPORTADO']
    )
)

print('\nRESUMEN DE COBERTURA POR SEXO')
print('Ofertas-período:', len(cobertura_sexo))
print(
    'Con Masculino y Femenino:',
    int(cobertura_sexo['MASCULINO_Y_FEMENINO'].sum())
)
print(
    'Con una categoría:',
    int((cobertura_sexo['NUM_CATEGORIAS'] == 1).sum())
)
print(
    'Con dos categorías:',
    int((cobertura_sexo['NUM_CATEGORIAS'] == 2).sum())
)
print(
    'Con tres categorías:',
    int((cobertura_sexo['NUM_CATEGORIAS'] == 3).sum())
)

# ============================================================
# 5. COBERTURA DE LAS CINCO VARIABLES POR PERÍODO
# ============================================================

# Se considera que una variable tiene información disponible
# cuando al menos una categoría de sexo reporta un valor.

cobertura_periodo = (
    df.groupby(claves_periodo, dropna=False)[variables_cluster]
    .agg(lambda x: x.notna().any())
    .reset_index()
)

for agrupacion, nombre in [
    ('ANO', 'AÑO'),
    (['ANO', 'SEMESTRE'], 'AÑO Y SEMESTRE')
]:
    grupo = cobertura_periodo.groupby(
        agrupacion
    )[variables_cluster]

    mostrar(
        f'OFERTAS-PERÍODO CON DATO POR {nombre}',
        grupo.sum()
    )

    mostrar(
        f'% COBERTURA POR {nombre}',
        (grupo.mean() * 100).round(2)
    )

# ============================================================
# 6. COBERTURA HISTÓRICA POR OFERTA
# ============================================================

cobertura_ofertas = (
    cobertura_periodo
    .groupby(claves_oferta, dropna=False)[variables_cluster]
    .sum()
    .astype(int)
    .add_suffix('_SEMESTRES')
)

cobertura_ofertas['PERIODOS_OBSERVADOS'] = (
    cobertura_periodo
    .groupby(claves_oferta, dropna=False)
    .size()
)

# ============================================================
# 7. DISPONIBILIDAD EN 2025
# ============================================================

cobertura_2025 = (
    cobertura_periodo[
        cobertura_periodo['ANO'] == 2025
    ]
    .groupby(claves_oferta, dropna=False)[variables_cluster]
    .any()
    .add_suffix('_2025')
)

cobertura_ofertas = (
    cobertura_ofertas
    .join(cobertura_2025)
    .reset_index()
)

columnas_2025 = [
    variable + '_2025'
    for variable in variables_cluster
]

mostrar(
    'SEMESTRES OBSERVADOS POR OFERTA',
    cobertura_ofertas['PERIODOS_OBSERVADOS']
    .value_counts()
    .sort_index()
    .to_frame('OFERTAS')
)

mostrar(
    'ESTADÍSTICOS DE COBERTURA HISTÓRICA',
    cobertura_ofertas[
        [
            variable + '_SEMESTRES'
            for variable in variables_cluster
        ]
    ].describe().round(2)
)

mostrar(
    'OFERTAS CON DATO EN 2025',
    cobertura_ofertas[columnas_2025]
    .sum()
    .to_frame('OFERTAS_CON_DATO')
)

sin_2025 = (
    cobertura_ofertas[columnas_2025]
    .isna()
    .all(axis=1)
    .sum()
)

print('Ofertas sin registros en 2025:', sin_2025)
print('Ofertas presentes en 2025:', n_ofertas - sin_2025)

# ============================================================
# 8. DISPONIBILIDAD DE DATOS SEGÚN SEXO
# ============================================================

for agrupacion, nombre in [
    ('SEXO', 'SEXO'),
    (['ANO', 'SEXO'], 'AÑO Y SEXO')
]:
    grupo = df.groupby(agrupacion)[variables_cluster]

    porcentaje = (
        grupo.count()
        .div(grupo.size(), axis=0)
        * 100
    ).round(2)

    mostrar(
        f'% DATOS DISPONIBLES POR {nombre}',
        porcentaje
    )

# ============================================================
# 9. MATRIZ FINAL DE COBERTURA
# ============================================================

mostrar(
    'COBERTURA POR OFERTA ACADÉMICA',
    cobertura_ofertas.head(15)
)

# ============================================================
# 10. VALIDACIONES DE INTEGRIDAD
# ============================================================

assert duplicados == 0, (
    'Existen registros duplicados según las claves definidas'
)

assert cobertura_sexo['NUM_CATEGORIAS'].sum() == len(df), (
    'El conteo de categorías no coincide con los registros'
)

assert len(cobertura_periodo) == len(cobertura_sexo), (
    'Inconsistencia entre cobertura temporal y por sexo'
)

assert len(cobertura_ofertas) == n_ofertas, (
    'Inconsistencia en el número de ofertas'
)

assert cobertura_ofertas[
    'PERIODOS_OBSERVADOS'
].between(1, 12).all(), (
    'Se encontraron ofertas con períodos fuera del rango esperado'
)

assert n_ofertas == 243, (
    f'Se esperaban 243 ofertas, se encontraron {n_ofertas}'
)

print('\n' + '=' * 60)
print('AUDITORÍA INTEGRAL FINALIZADA CORRECTAMENTE')
print('=' * 60)
print('La base df permanece intacta.')
print('No se eliminaron registros ni se imputaron valores.')


#Carga de base de datos

In [ ]:

# ============================================================
# PROYECTO SNIES 2020-2025
# CARGAR BASE DE DATOS ESTANDARIZADA
# ============================================================

import pandas as pd
from pathlib import Path
from google.colab import drive
from IPython.display import display

# ============================================================
# 1. CONECTAR GOOGLE DRIVE
# ============================================================

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# ============================================================
# 2. DEFINIR RUTA DE LA BASE ESTANDARIZADA
# ============================================================

ruta = (
    '/content/drive/MyDrive/MaestriaUPB_Mineria_Datos/'
    'P_final/SNIES_2020_2025_BASE_ESTANDARIZADA.csv'
)

# ============================================================
# 3. DEFINIR IDENTIFICADORES
# ============================================================

claves_oferta = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

# ============================================================
# 4. CARGAR BASE ESTANDARIZADA
# ============================================================

df = pd.read_csv(
    ruta,
    encoding='utf-8-sig',
    dtype={col: str for col in claves_oferta},
    low_memory=False
)

# ============================================================
# 5. VERIFICAR INTEGRIDAD
# ============================================================

assert df.shape == (3896, 60), (
    f'Dimensiones incorrectas: {df.shape}'
)

assert df[claves_oferta].drop_duplicates().shape[0] == 243, (
    'El número de ofertas académicas no coincide'
)

assert df.duplicated(
    subset=claves_oferta + ['ANO', 'SEMESTRE', 'SEXO']
).sum() == 0, (
    'Existen registros duplicados'
)

# ============================================================
# 6. PRESENTAR RESULTADOS
# ============================================================

print('=' * 60)
print('BASE SNIES ESTANDARIZADA CARGADA CORRECTAMENTE')
print('=' * 60)

print('Dimensiones:', df.shape)
print('Ofertas académicas:',
      df[claves_oferta].drop_duplicates().shape[0])

print('\nDISTRIBUCIÓN POR SEXO')
display(df['SEXO'].value_counts(dropna=False))

print('\nPRIMEROS 10 REGISTROS')
display(df.head(10))

print('\nBASE ESTANDARIZADA LISTA PARA CONTINUAR EL PROCESO')


# CLUSTERING DE PROGRAMAS DE INGENIERÍA — SNIES 2020–2025

## 1. Objetivo

Identificar perfiles de demanda académica de las ofertas de programas de ingeniería, considerando su situación en 2025 y su evolución durante el periodo 2020–2025, mediante técnicas de aprendizaje no supervisado que apoyen el análisis de pertinencia académica y la identificación de oportunidades para fortalecer la participación femenina.

## 2. Base de datos

El estudio utiliza información del Sistema Nacional de Información de la Educación Superior (SNIES), correspondiente al periodo 2020–2025.

La información fue integrada, estandarizada y preparada a partir de los registros institucionales disponibles, conservando la diferenciación por año, semestre y sexo.

La unidad de análisis es la **oferta académica**, identificada mediante la combinación de cuatro variables:

- `CODIGO SNIES DEL PROGRAMA`
- `CODIGO DE LA INSTITUCION`
- `CODIGO DEL MUNICIPIO (PROGRAMA)`
- `ID MODALIDAD`

Esta definición permite distinguir ofertas de un mismo programa académico cuando corresponden a distintas instituciones, municipios o modalidades.

La base preparada conserva la información original necesaria para caracterizar posteriormente las ofertas según programa, institución, localización y participación femenina.

## 3. Preparación y auditoría de datos

Se realizó la integración, estandarización y auditoría de siete categorías de información SNIES:

- Inscritos.
- Admitidos.
- Matriculados de primer curso.
- Matriculados.
- Graduados.
- Docentes.
- Administrativos.

Durante esta etapa se revisaron la integridad de los registros, la consistencia de las variables, la presencia de duplicados, la disponibilidad temporal y la correspondencia de los valores integrados con sus fuentes.

### Decisiones metodológicas

**Tratamiento de valores faltantes:** se conservaron los valores `NaN` en la base preparada, sin imputaciones ni sustituciones automáticas por cero. Para el clustering se establecieron criterios de elegibilidad que permitieron construir matrices analíticas completas.

**Diferenciación semestral:** se mantuvieron separados los semestres S1 y S2, con el propósito de conservar las particularidades del comportamiento de la demanda académica.

**Proporciones superiores a uno:** se conservaron los valores originales de los indicadores de admisión e ingreso, sin truncamientos ni eliminación automática de observaciones.

**Balanceo de datos:** no se aplicaron técnicas de balanceo de clases, dado que K-means es un algoritmo de aprendizaje no supervisado y no existe una variable objetivo.

## 4. Diseño de los escenarios de clustering

Se definieron tres escenarios complementarios a partir de la misma base preparada.

| Escenario | Propósito | Enfoque |
|---|---|---|
| **A. Demanda reciente** | Caracterizar la demanda académica durante 2025 | Situación actual de las ofertas |
| **B. Evolución histórica** | Caracterizar la evolución de la demanda entre 2020 y 2025 | Promedios y tendencias históricas |
| **C. Demanda integral** | Combinar información reciente e histórica | Situación actual y trayectoria de la demanda |

### 4.1. Escenario A. Demanda reciente

Se consideraron ofertas académicas con información completa de inscritos y matriculados de primer curso durante los dos semestres de 2025, incluyendo los registros correspondientes a ambos sexos.

Se construyeron indicadores de:

- Inscritos por semestre.
- Matriculados de primer curso por semestre.
- Proporción de admisión.
- Proporción de ingreso de admitidos.
- Proporción de ingreso de inscritos.

### 4.2. Escenario B. Evolución histórica

Se consideraron ofertas con al menos cuatro años completos de información durante 2020–2025, conservando la diferenciación semestral.

Se construyeron características de:

- Promedio histórico de inscritos.
- Tendencia histórica de inscritos.
- Promedio histórico de matriculados de primer curso.
- Tendencia histórica de matriculados de primer curso.
- Promedios históricos de las proporciones de admisión e ingreso.

Las tendencias se representaron mediante pendientes estimadas por regresión lineal en función del año, expresadas en estudiantes por año.

### 4.3. Escenario C. Demanda integral

Se integraron las características recientes e históricas de las ofertas que cumplían simultáneamente los criterios de elegibilidad de los escenarios A y B.

Este escenario permite analizar conjuntamente la situación reciente y la trayectoria histórica de la demanda.

## 5. Análisis exploratorio de datos

Para cada escenario se desarrolló un análisis exploratorio que incluyó:

- Estadísticas descriptivas de las variables cuantitativas.
- Histogramas.
- Diagramas de caja.
- Distribuciones de frecuencia de las variables categóricas.
- Reportes automatizados mediante `ydata-profiling`.
- Revisión y corrección de los tipos de datos.

El análisis permitió examinar las distribuciones, la dispersión, los valores extremos y las características institucionales de las ofertas seleccionadas.

## 6. Transformación de variables

### 6.1. Normalización

Se utilizó `MinMaxScaler` de Scikit-learn para normalizar las variables cuantitativas al intervalo [0, 1].

Cada escenario se normalizó de manera independiente y se conservaron los parámetros de los escaladores para recuperar posteriormente la escala original de los centroides.

### 6.2. Codificación de variables categóricas

Se utilizó `pd.get_dummies()` para transformar las variables categóricas seleccionadas para el clustering:

- `SECTOR IES`
- `TIPO IES`

Debido a que ambas variables presentan dos categorías, se aplicó `drop_first=True`, conservando una variable dummy por cada característica.

La variable `MODALIDAD` se reservó para la caracterización posterior y se excluyó del cálculo de distancias.

Los cuatro códigos que identifican cada oferta también fueron excluidos del entrenamiento, pero se conservaron para permitir la identificación de los programas académicos.

## 7. Aprendizaje del modelo

Se aplicó el algoritmo **K-means** de manera independiente a cada uno de los tres escenarios.

El modelo utilizó las variables cuantitativas normalizadas y las variables dummies seleccionadas.

El entrenamiento se realizó con la totalidad de las ofertas académicas elegibles de cada escenario, sin división entre entrenamiento y prueba.

### 7.1. Determinación del número de clústeres

Se utilizaron dos procedimientos:

**Método del codo:** evaluación de la inercia o suma de cuadrados dentro de los clústeres (SSE) para diferentes valores de k.

**Método de la silueta:** evaluación de la cohesión interna y separación de los grupos para diferentes valores de k.

Los valores de k seleccionados para el entrenamiento correspondieron a los máximos del coeficiente de silueta dentro del rango evaluado.

### 7.2. Configuración de K-means

Se utilizaron los siguientes parámetros:

- `max_iter = 300`
- `random_state = 42`
- `n_init = 10`

Se entrenó un modelo independiente por escenario y se conservaron los estimadores ajustados para su posterior evaluación e interpretación.

## 8. Evaluación del modelo

La evaluación se desarrolló mediante dos indicadores.

**Inercia (SSE):** cuantifica la suma de las distancias cuadráticas de las observaciones a los centroides de sus respectivos clústeres. Permite examinar la compactación interna de las agrupaciones.

**Coeficiente de silueta:** evalúa simultáneamente la cohesión de las observaciones dentro de su clúster y la separación respecto a otros grupos.

Se consideraron preferibles valores positivos de silueta y, como referencia metodológica, valores superiores a 0.5.

La interpretación de los indicadores se realizó teniendo en cuenta que los escenarios contienen diferentes conjuntos de ofertas y características, por lo que las inercias no constituyen una base de comparación directa entre escenarios.

## 9. Perfilamiento de los clústeres

Los centroides fueron calculados automáticamente por K-means durante el entrenamiento y recuperados mediante `cluster_centers_`.

Posteriormente se organizaron en DataFrames de Pandas, identificando las variables correspondientes a cada componente.

Para facilitar su interpretación, se aplicó la transformación inversa de `MinMaxScaler`, recuperando las unidades originales de las características cuantitativas.

### Criterios de interpretación

El perfilamiento consideró:

- Volumen de inscritos.
- Volumen de matriculados de primer curso.
- Proporción de admisión.
- Proporción de ingreso de admitidos.
- Ingreso relativo.
- Tendencias históricas por semestre.
- Características institucionales.

**Ingreso relativo** se definió como el cociente entre matriculados de primer curso e inscritos:

**Ingreso relativo = Matriculados de primer curso / Inscritos**

Este indicador describe la relación agregada entre la demanda registrada y la matrícula de primer curso. No debe confundirse con la proporción de admisión ni interpretarse necesariamente como una probabilidad individual de matrícula.

Las denominaciones de los perfiles se establecieron a partir de las características de los centroides desnormalizados.

Los términos demanda alta, intermedia o baja se utilizaron comparativamente dentro de cada escenario, sin establecer umbrales institucionales universales.

Para los escenarios históricos se consideraron las pendientes de inscritos y matriculados de primer curso, manteniendo la diferenciación entre semestres.

## 10. Identificación de ofertas académicas por clúster

Una vez ajustados los modelos se recuperaron las etiquetas mediante `model.labels_` y se asociaron a las ofertas de cada escenario.

Se conservaron los identificadores originales para relacionar posteriormente cada agrupación con:

- Programa académico.
- Institución de educación superior.
- Municipio de oferta.
- Modalidad.

Esta asociación permite identificar las ofertas que integran cada perfil.

Debe tenerse en cuenta que los centroides representan comportamientos promedio. Por tanto, para determinar si una oferta individual presenta crecimiento, estabilidad o disminución es necesario revisar sus pendientes históricas particulares.

## 11. Participación femenina y pertinencia académica

La información por sexo se conservó en la base preparada, pero no se incorporó como variable explicativa del clustering.

Esta decisión permite identificar los perfiles de demanda académica sin que la composición por sexo determine directamente la formación de los grupos.

La caracterización posterior de las ofertas permitirá analizar la participación femenina según programa, institución, territorio y perfil de demanda.

Este análisis podrá servir de base para identificar oportunidades de promoción académica, orientación vocacional y estrategias institucionales orientadas al fortalecimiento de la participación de mujeres en programas de ingeniería.

## 12. Alcance del estudio

El estudio se estructuró mediante **tres escenarios complementarios de clustering**, aplicando el algoritmo K-means.

La metodología permitió abordar la demanda académica desde tres perspectivas:

- Situación reciente de las ofertas.
- Evolución histórica.
- Integración de la situación reciente y la trayectoria histórica.

El propósito del perfilamiento es generar información interpretable para apoyar el análisis de pertinencia académica y la identificación de oportunidades de fortalecimiento de la oferta.

La clasificación de cada programa como creciente, estable o decreciente requiere complementar los perfiles de clúster con la revisión de las tendencias individuales de las ofertas académicas.

In [ ]:

# ============================================================
# SNIES 2020-2025 | MATRICES DEFINITIVAS PARA CLUSTERING
# SELECCIÓN, ELEGIBILIDAD Y TRATAMIENTO DE FALTANTES
#
# Entrada: df (base estandarizada de 3896 x 60)
# Salidas: matriz_A, matriz_B, matriz_C
#
# Sin crear otra base CSV.
# Sin matrices intermedias de 40 o 220 columnas.
# Sin imputaciones, normalización ni dummies.
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

claves_oferta = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

indicadores = [
    'INSCRITOS',
    'MATRICULADOS PRIMER CURSO'
]

razones = {
    'PROP_ADMISION': (
        'ADMITIDOS',
        'INSCRITOS'
    ),
    'PROP_INGRESO_ADMITIDOS': (
        'MATRICULADOS PRIMER CURSO',
        'ADMITIDOS'
    ),
    'PROP_INGRESO_INSCRITOS': (
        'MATRICULADOS PRIMER CURSO',
        'INSCRITOS'
    )
}

categoricas = [
    'SECTOR IES',
    'TIPO IES',
    'MODALIDAD'
]

MIN_ANIOS = 4

# Validar que se parte de la base estandarizada
assert df.shape == (3896, 60), (
    f'Se esperaba la base estandarizada (3896, 60), '
    f'pero se encontró {df.shape}'
)

columnas_requeridas = (
    claves_oferta +
    ['ANO', 'SEMESTRE', 'SEXO', 'ADMITIDOS'] +
    indicadores +
    categoricas
)

faltantes_columnas = [
    c for c in columnas_requeridas
    if c not in df.columns
]

assert not faltantes_columnas, (
    f'Faltan columnas requeridas: {faltantes_columnas}'
)

# ------------------------------------------------------------
# 2. SELECCIÓN Y ELEGIBILIDAD
# ------------------------------------------------------------

# Seleccionar exclusivamente registros Masculino y Femenino
# para construir las matrices.
# La base estandarizada df permanece intacta.

sexo_normalizado = (
    df['SEXO']
    .astype('string')
    .str.strip()
    .str.upper()
)

datos_cluster = df.loc[
    sexo_normalizado.isin(['MASCULINO', 'FEMENINO'])
].copy()

assert datos_cluster.shape == (3889, 60), (
    f'Registros seleccionados inesperados: '
    f'{datos_cluster.shape}'
)

# Cada oferta-período debe contener ambos sexos
# y valores disponibles de las dos variables centrales.

grupo = datos_cluster.groupby(
    claves_oferta + ['ANO', 'SEMESTRE'],
    observed=True,
    dropna=False
)

completo = (
    grupo[indicadores].count().eq(2).all(axis=1)
    & grupo['SEXO'].nunique().eq(2)
)

cobertura_anual = (
    completo
    .groupby(level=claves_oferta + ['ANO'])
    .sum()
    .reset_index(name='SEMESTRES_COMPLETOS')
)

elegibilidad = (
    cobertura_anual.groupby(claves_oferta)
    .agg(
        ANIOS_COMPLETOS=(
            'SEMESTRES_COMPLETOS',
            lambda x: x.eq(2).sum()
        )
    )
    .reset_index()
)

reciente = cobertura_anual.loc[
    cobertura_anual['ANO'].eq(2025),
    claves_oferta + ['SEMESTRES_COMPLETOS']
].rename(
    columns={'SEMESTRES_COMPLETOS': 'SEMESTRES_2025'}
)

elegibilidad = elegibilidad.merge(
    reciente,
    on=claves_oferta,
    how='left',
    validate='one_to_one'
)

elegibilidad['A'] = (
    elegibilidad['SEMESTRES_2025'].eq(2)
)

elegibilidad['B'] = (
    elegibilidad['ANIOS_COMPLETOS'].ge(MIN_ANIOS)
)

elegibilidad['C'] = (
    elegibilidad['A'] & elegibilidad['B']
)

ofertas_A = elegibilidad.loc[
    elegibilidad['A'], claves_oferta
]

ofertas_B = elegibilidad.loc[
    elegibilidad['B'], claves_oferta
]

ofertas_C = elegibilidad.loc[
    elegibilidad['C'], claves_oferta
]

# ------------------------------------------------------------
# 3. TOTALES SEMESTRALES Y PROPORCIONES
# ------------------------------------------------------------

# Se suman los registros Masculino y Femenino.
# min_count=2 exige información para ambos.
# No se imputan valores ni se limitan proporciones a 1.

variables_conteo = [
    'INSCRITOS',
    'ADMITIDOS',
    'MATRICULADOS PRIMER CURSO'
]

totales = (
    grupo[variables_conteo]
    .sum(min_count=2)
    .reset_index()
)

for nombre, (numerador, denominador) in razones.items():

    totales[nombre] = (
        totales[numerador] /
        totales[denominador].replace(0, np.nan)
    )

variables_periodo = indicadores + list(razones)

# ------------------------------------------------------------
# 4. ESCENARIO A: DEMANDA RECIENTE 2025
# ------------------------------------------------------------

# Incluye:
# - Inscritos S1 y S2
# - Matriculados primer curso S1 y S2
# - Tres proporciones para ambos semestres

def construir_reciente(ofertas):

    indice = pd.MultiIndex.from_frame(ofertas)

    datos = (
        totales.loc[totales['ANO'].eq(2025)]
        .pivot(
            index=claves_oferta,
            columns='SEMESTRE',
            values=variables_periodo
        )
        .reindex(indice)
    )

    columnas = pd.MultiIndex.from_product([
        variables_periodo,
        [1, 2]
    ])

    datos = datos.reindex(columns=columnas)

    datos.columns = [
        f'{variable}_2025_S{semestre}'
        for variable, semestre in datos.columns
    ]

    return datos

# ------------------------------------------------------------
# 5. ESCENARIO B: EVOLUCIÓN HISTÓRICA 2020-2025
# ------------------------------------------------------------

# Se utilizan exclusivamente años con S1 y S2 completos.
#
# Para inscritos y matriculados de primer curso:
# - Promedio histórico por semestre
# - Pendiente anual por semestre
#
# Para las proporciones:
# - Promedio histórico por semestre

anios_validos = cobertura_anual.loc[
    cobertura_anual['SEMESTRES_COMPLETOS'].eq(2),
    claves_oferta + ['ANO']
]

historicos = (
    totales
    .merge(
        anios_validos,
        on=claves_oferta + ['ANO'],
        how='inner',
        validate='many_to_one'
    )
    .merge(
        ofertas_B,
        on=claves_oferta,
        how='inner',
        validate='many_to_one'
    )
)

def construir_historico(datos, ofertas):

    registros = []

    for clave, grupo_oferta in datos.groupby(
        claves_oferta,
        sort=False,
        dropna=False
    ):

        registro = dict(zip(claves_oferta, clave))

        for semestre in [1, 2]:

            periodo = grupo_oferta.loc[
                grupo_oferta['SEMESTRE'].eq(semestre)
            ].sort_values('ANO')

            x = periodo['ANO'].to_numpy(dtype=float)

            for variable in indicadores:

                y = periodo[variable].to_numpy(dtype=float)

                validos = (
                    np.isfinite(x) &
                    np.isfinite(y)
                )

                registro[
                    f'{variable}_PROM_S{semestre}'
                ] = (
                    y[validos].mean()
                    if validos.any()
                    else np.nan
                )

                registro[
                    f'{variable}_PENDIENTE_S{semestre}'
                ] = (
                    np.polyfit(
                        x[validos],
                        y[validos],
                        1
                    )[0]
                    if validos.sum() >= 2
                    else np.nan
                )

            for proporcion in razones:

                registro[
                    f'{proporcion}_PROM_S{semestre}'
                ] = periodo[proporcion].mean()

        registros.append(registro)

    return (
        pd.DataFrame(registros)
        .set_index(claves_oferta)
        .reindex(pd.MultiIndex.from_frame(ofertas))
    )

# ------------------------------------------------------------
# 6. CONSTRUIR LOS TRES ESCENARIOS
# ------------------------------------------------------------

resultado_A = construir_reciente(ofertas_A)

resultado_B = construir_historico(
    historicos,
    ofertas_B
)

indice_C = pd.MultiIndex.from_frame(ofertas_C)

resultado_C = (
    resultado_B.reindex(indice_C)
    .join(
        resultado_A.reindex(indice_C),
        validate='one_to_one'
    )
)

# ------------------------------------------------------------
# 7. INCORPORAR VARIABLES INSTITUCIONALES
# ------------------------------------------------------------

# Se recuperan desde la misma población seleccionada.
# No se crean dummies en esta etapa.

contexto = (
    datos_cluster
    .sort_values(['ANO', 'SEMESTRE'])
    .drop_duplicates(claves_oferta, keep='last')
    .set_index(claves_oferta)[categoricas]
)

def finalizar(datos, escenario, dimensiones):

    datos = datos.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Se exige completitud de las características numéricas.
    # No se imputan valores.

    faltantes = datos.isna().any(axis=1)

    if faltantes.any():
        raise ValueError(
            f'Escenario {escenario}: '
            f'{int(faltantes.sum())} ofertas tienen '
            'características faltantes.'
        )

    datos = datos.join(
        contexto,
        how='left',
        validate='one_to_one'
    )

    if datos[categoricas].isna().any().any():
        raise ValueError(
            f'Escenario {escenario}: '
            'faltantes en variables institucionales.'
        )

    for columna in categoricas:
        datos[columna] = (
            datos[columna].astype('string')
        )

    datos = datos.reset_index()

    if datos.shape != dimensiones:
        raise ValueError(
            f'Escenario {escenario}: '
            f'dimensiones inesperadas {datos.shape}; '
            f'se esperaban {dimensiones}.'
        )

    if datos.duplicated(claves_oferta).any():
        raise ValueError(
            f'Escenario {escenario}: ofertas duplicadas.'
        )

    return datos

# ------------------------------------------------------------
# MATRICES DEFINITIVAS
# ------------------------------------------------------------

# Solo se asignan después de validar los tres escenarios.

salidas = {
    'A': finalizar(resultado_A, 'A', (124, 17)),
    'B': finalizar(resultado_B, 'B', (115, 21)),
    'C': finalizar(resultado_C, 'C', (105, 31))
}

matriz_A = salidas['A']
matriz_B = salidas['B']
matriz_C = salidas['C']

# ------------------------------------------------------------
# 8. REPORTE FINAL
# ------------------------------------------------------------

print('=' * 70)
print('SNIES | MATRICES DEFINITIVAS PARA CLUSTERING')
print('=' * 70)

print('Base estandarizada:', df.shape)
print('Registros seleccionados:', datos_cluster.shape)
print('Registros no incluidos:', len(df) - len(datos_cluster))

display(pd.DataFrame([
    {
        'ESCENARIO': nombre,
        'OFERTAS': len(matriz),
        'NUMERICAS': len(
            matriz.select_dtypes(include=np.number).columns
        ),
        'CATEGORICAS': len(categoricas),
        'FALTANTES': int(
            matriz.isna().sum().sum()
        ),
        'PROP_MAYOR_1': int(
            matriz.filter(regex=r'^PROP_')
            .gt(1)
            .any(axis=1)
            .sum()
        )
    }
    for nombre, matriz in [
        ('A', matriz_A),
        ('B', matriz_B),
        ('C', matriz_C)
    ]
]))

print('\nMATRICES DEFINITIVAS:')
print('matriz_A:', matriz_A.shape)
print('matriz_B:', matriz_B.shape)
print('matriz_C:', matriz_C.shape)

print('\nLISTAS PARA:')
print('1. Descripción estadística')
print('2. Histogramas y boxplots')
print('3. ProfileReport')
print('4. Normalización y dummies')
print('5. K-means')

print('\nLa base df permanece intacta.')
print('No se imputaron valores faltantes.')


In [ ]:

# ============================================================
# SNIES | HISTOGRAMAS Y BOXPLOTS
# MATRICES DEFINITIVAS - ESCENARIOS A, B Y C
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

matrices = {
    'A - Demanda 2025': matriz_A,
    'B - Evolución histórica': matriz_B,
    'C - Demanda integral': matriz_C
}

for escenario, matriz in matrices.items():

    # Seleccionar solo las variables numéricas del clustering
    columnas = matriz.select_dtypes(include=np.number).columns

    print(f"\n{'=' * 65}")
    print(f"ESCENARIO {escenario}")
    print(f"Ofertas: {len(matriz)} | Variables: {len(columnas)}")
    print('=' * 65)

    # Una figura por variable: histograma y boxplot
    for variable in columnas:

        fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))

        sns.histplot(
            data=matriz,
            x=variable,
            bins=20,
            kde=True,
            ax=ax[0]
        )

        ax[0].set_title('Histograma')
        ax[0].set_xlabel(variable)
        ax[0].set_ylabel('Frecuencia')

        sns.boxplot(
            data=matriz,
            x=variable,
            ax=ax[1]
        )

        ax[1].set_title('Boxplot')
        ax[1].set_xlabel(variable)

        fig.suptitle(
            f'{escenario} | {variable}',
            fontsize=11,
            fontweight='bold'
        )

        plt.tight_layout()
        plt.show()
        plt.close(fig)

print("\nHistogramas y boxplots finalizados.")
print("Las matrices A, B y C permanecen intactas.")


In [ ]:

# ============================================================
# SNIES | DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS
# MATRICES DEFINITIVAS - ESCENARIOS A, B Y C
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

categoricas = ['SECTOR IES', 'TIPO IES', 'MODALIDAD']

for escenario, matriz in {
    'A - Demanda 2025': matriz_A,
    'B - Evolución histórica': matriz_B,
    'C - Demanda integral': matriz_C
}.items():

    print(f"\n{'=' * 65}")
    print(f"ESCENARIO {escenario} | {len(matriz)} ofertas")
    print('=' * 65)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, variable in zip(axes, categoricas):

        frecuencias = (
            matriz[variable]
            .value_counts(dropna=False)
            .sort_values(ascending=True)
        )

        sns.barplot(
            x=frecuencias.values,
            y=frecuencias.index.astype(str),
            ax=ax,
            color='steelblue'
        )

        ax.set_title(variable, fontweight='bold')
        ax.set_xlabel('Número de ofertas')
        ax.set_ylabel('')

        ax.set_xlim(0, max(frecuencias.values) * 1.25)

        for i, frecuencia in enumerate(frecuencias.values):
            porcentaje = frecuencia / len(matriz) * 100

            ax.text(
                frecuencia + max(frecuencias.values) * 0.02,
                i,
                f'{frecuencia} ({porcentaje:.1f}%)',
                va='center',
                fontsize=9
            )

    fig.suptitle(
        f'SNIES | {escenario}',
        fontsize=14,
        fontweight='bold'
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

print("\nDistribuciones categóricas finalizadas.")
print("Las matrices A, B y C permanecen intactas.")


In [ ]:
# Cargar librería para Profiling
!pip install ydata-profiling

In [ ]:

# ============================================================
# SNIES | PROFILE REPORT
# MATRICES DEFINITIVAS - ESCENARIOS A, B Y C
# ============================================================

from ydata_profiling import ProfileReport
from IPython.display import display, FileLink
from pathlib import Path

# Carpeta para guardar los informes
carpeta = Path(
    '/content/drive/MyDrive/MaestriaUPB_Mineria_Datos/'
    'P_final/Perfilados'
)
carpeta.mkdir(parents=True, exist_ok=True)

# Perfilamiento de los tres escenarios
for escenario, matriz in {
    'A': matriz_A,
    'B': matriz_B,
    'C': matriz_C
}.items():

    print(f"\n{'=' * 60}")
    print(f"PROFILE REPORT | ESCENARIO {escenario}")
    print(f"Ofertas: {len(matriz)} | Variables: {matriz.shape[1]}")
    print('=' * 60)

    perfil = ProfileReport(
        matriz,
        title=f'SNIES 2020-2025 | Escenario {escenario}',
        explorative=True,
        progress_bar=False
    )

    archivo = carpeta / f'ProfileReport_SNIES_{escenario}.html'
    perfil.to_file(str(archivo))

    print("Informe generado:")
    display(FileLink(str(archivo)))

print("\nPerfilamiento de los tres escenarios finalizado.")
print("Las matrices A, B y C no fueron modificadas.")


# **Transformaciones: normalizar y crear dummies**

#No se hace balabece por que no hay variable objetivo

In [ ]:

# ============================================================
# SNIES | NORMALIZACIÓN DE VARIABLES NUMÉRICAS
# MINMAXSCALER - ESCENARIOS A, B Y C
#
# Se conservan las matrices originales.
# Se crean matrices normalizadas independientes.
# ============================================================

from sklearn.preprocessing import MinMaxScaler
import numpy as np
from IPython.display import display

# 1. Matrices originales
matrices = {
    'A': matriz_A,
    'B': matriz_B,
    'C': matriz_C
}

# 2. Almacenar escaladores y matrices normalizadas
escaladores = {}
matrices_normalizadas = {}
variables_numericas = {}

# 3. Normalizar cada escenario
for escenario, matriz in matrices.items():

    # Seleccionar variables numéricas del clustering
    variables_a_normalizar = matriz.select_dtypes(
        include=np.number
    ).columns.tolist()

    # Crear copia independiente
    matriz_norm = matriz.copy(deep=True)

    # Crear y ajustar el escalador
    min_max_scaler = MinMaxScaler()

    matriz_norm[variables_a_normalizar] = (
        min_max_scaler.fit_transform(
            matriz[variables_a_normalizar]
        )
    )

    # Guardar resultados
    escaladores[escenario] = min_max_scaler
    matrices_normalizadas[escenario] = matriz_norm
    variables_numericas[escenario] = variables_a_normalizar

    # Validaciones
    assert matriz_norm.shape == matriz.shape

    assert (
        matriz_norm[variables_a_normalizar]
        .ge(0).all().all()
    )

    assert (
        matriz_norm[variables_a_normalizar]
        .le(1 + 1e-10).all().all()
    )

    print(f"\n{'=' * 65}")
    print(f"ESCENARIO {escenario}")
    print(f"Variables normalizadas: {len(variables_a_normalizar)}")
    print("=" * 65)

    display(matriz_norm.head())

# 4. Crear variables individuales
matriz_A_norm = matrices_normalizadas['A']
matriz_B_norm = matrices_normalizadas['B']
matriz_C_norm = matrices_normalizadas['C']

print("\nNORMALIZACIÓN FINALIZADA")
print("Matrices originales: CONSERVADAS")
print("Matrices normalizadas: A, B y C")
print("Escaladores: A, B y C")
print("Identificadores y variables categóricas: SIN MODIFICAR")


In [ ]:

# ============================================================
# SNIES 2020-2025 | GENERACIÓN DE VARIABLES DUMMY
# ESCENARIOS A, B Y C
#
# Entrada:
#   matriz_A_norm
#   matriz_B_norm
#   matriz_C_norm
#
# Salida:
#   matriz_A_modelo
#   matriz_B_modelo
#   matriz_C_modelo
#
# Conserva matrices originales y normalizadas.
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

variables_dummy = [
    'SECTOR IES',
    'TIPO IES'
]

matrices_normalizadas = {
    'A': matriz_A_norm,
    'B': matriz_B_norm,
    'C': matriz_C_norm
}

matrices_modelo = {}

# ============================================================
# 2. GENERAR VARIABLES DUMMY
# ============================================================

for escenario, matriz in matrices_normalizadas.items():

    # Verificar las variables categóricas necesarias
    assert all(
        columna in matriz.columns
        for columna in variables_dummy
    ), (
        f'Escenario {escenario}: faltan '
        'variables categóricas.'
    )

    # Copia independiente
    matriz_trabajo = matriz.copy(deep=True)

    # Codificación binaria de las variables indicadas
    matriz_transformada = pd.get_dummies(
        matriz_trabajo,
        columns=variables_dummy,
        drop_first=True,
        dtype=int
    )

    # Guardar matriz transformada
    matrices_modelo[escenario] = matriz_transformada

    # Identificar columnas dummy creadas
    columnas_dummy = [
        columna
        for columna in matriz_transformada.columns
        if columna.startswith((
            'SECTOR IES_',
            'TIPO IES_'
        ))
    ]

    # Validar integridad
    assert len(matriz_transformada) == len(matriz)

    assert not matriz_transformada.isna().any().any(), (
        f'Escenario {escenario}: se detectaron faltantes.'
    )

    print(f"\n{'=' * 65}")
    print(f"ESCENARIO {escenario}")
    print("=" * 65)

    print("Dimensiones:", matriz_transformada.shape)
    print("Variables dummy:", len(columnas_dummy))
    print("Columnas dummy generadas:", columnas_dummy)

    display(matriz_transformada.head())

# ============================================================
# 3. ASIGNAR MATRICES PARA MODELADO
# ============================================================

matriz_A_modelo = matrices_modelo['A']
matriz_B_modelo = matrices_modelo['B']
matriz_C_modelo = matrices_modelo['C']

# ============================================================
# 4. RESUMEN
# ============================================================

print("\n" + "=" * 65)
print("SNIES | CODIFICACIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 65)

print("Matrices originales: CONSERVADAS")
print("Matrices normalizadas: CONSERVADAS")
print("Matrices para modelado: GENERADAS")

print("\nESCENARIO A:", matriz_A_modelo.shape)
print("ESCENARIO B:", matriz_B_modelo.shape)
print("ESCENARIO C:", matriz_C_modelo.shape)

print("\nVARIABLES CODIFICADAS:")
print(variables_dummy)

print("\nMODALIDAD: conservada como información contextual.")
print("Matrices preparadas para seleccionar las variables de K-means.")


# 2. Aprendizaje del Modelo
* Metodo del codo
* Aplicar kmeans

In [ ]:

# ============================================================
# SNIES | APRENDIZAJE DEL MODELO
# MÉTODO DEL CODO - ESCENARIOS A, B Y C
# MATRICES NORMALIZADAS + VARIABLES DUMMY
# ============================================================

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# 1. IDENTIFICADORES Y VARIABLES CONTEXTUALES
# ============================================================

identificadores = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

columnas_excluir = identificadores + ['MODALIDAD']

# Matrices preparadas para modelado
matrices = {
    'A': matriz_A_modelo,
    'B': matriz_B_modelo,
    'C': matriz_C_modelo
}

# Número esperado de características
caracteristicas_esperadas = {
    'A': 12,
    'B': 16,
    'C': 26
}

# ============================================================
# 2. MÉTODO DEL CODO
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 4.5)
)

resultados_codo = {}

for ax, (escenario, matriz) in zip(
    axes,
    matrices.items()
):

    # Seleccionar exclusivamente las características
    # utilizadas para el clustering
    X = matriz.drop(
        columns=columnas_excluir
    )

    # Validar que todas sean numéricas
    assert all(
        pd.api.types.is_numeric_dtype(X[columna])
        for columna in X.columns
    ), (
        f'Escenario {escenario}: '
        'se encontraron variables no numéricas.'
    )

    assert X.shape[1] == caracteristicas_esperadas[escenario], (
        f'Escenario {escenario}: '
        f'se esperaban '
        f'{caracteristicas_esperadas[escenario]} '
        f'características y se obtuvieron {X.shape[1]}.'
    )

    assert np.isfinite(X.to_numpy(dtype=float)).all(), (
        f'Escenario {escenario}: '
        'se encontraron valores no finitos.'
    )

    # Evaluar k = 1 hasta k = 10
    valores_k = range(1, 11)
    inercias = []

    for k in valores_k:

        modelo = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        modelo.fit(X)

        inercias.append(modelo.inertia_)

    resultados_codo[escenario] = pd.DataFrame({
        'k': list(valores_k),
        'SSE': inercias
    })

    # Graficar resultados
    ax.plot(
        list(valores_k),
        inercias,
        marker='o'
    )

    ax.set_title(f'Escenario {escenario}')
    ax.set_xlabel('Número de clústeres (k)')
    ax.set_ylabel('Inercia (SSE)')
    ax.set_xticks(list(valores_k))
    ax.grid(alpha=0.3)

    print(
        f'Escenario {escenario}: '
        f'{X.shape[0]} ofertas | '
        f'{X.shape[1]} características'
    )

plt.suptitle(
    'SNIES | Método del codo',
    fontsize=14
)

plt.tight_layout()
plt.show()

print('\nMétodo del codo finalizado.')
print('Las matrices originales permanecen intactas.')


In [ ]:

# ============================================================
# SNIES | APRENDIZAJE DEL MODELO
# MÉTODO DE LA SILUETA - ESCENARIOS A, B Y C
# MATRICES NORMALIZADAS + VARIABLES DUMMY
# ============================================================

from sklearn.cluster import KMeans
from sklearn import metrics
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# 1. IDENTIFICADORES Y VARIABLES CONTEXTUALES
# ============================================================

identificadores = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

columnas_excluir = identificadores + ['MODALIDAD']

# Matrices preparadas para modelado
matrices = {
    'A': matriz_A_modelo,
    'B': matriz_B_modelo,
    'C': matriz_C_modelo
}

caracteristicas_esperadas = {
    'A': 12,
    'B': 16,
    'C': 26
}

# ============================================================
# 2. EVALUACIÓN MEDIANTE SILUETA
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

resultados_silueta = {}
mejores_k = {}

for ax, (escenario, matriz) in zip(
    axes,
    matrices.items()
):

    # Seleccionar variables utilizadas por K-means
    X = matriz.drop(columns=columnas_excluir)

    # Validaciones previas
    assert X.shape[1] == caracteristicas_esperadas[escenario], (
        f'Escenario {escenario}: '
        f'cantidad inesperada de características: {X.shape[1]}'
    )

    assert all(
        pd.api.types.is_numeric_dtype(X[col])
        for col in X.columns
    ), f'Escenario {escenario}: existen variables no numéricas.'

    assert np.isfinite(X.to_numpy(dtype=float)).all(), (
        f'Escenario {escenario}: existen valores no finitos.'
    )

    # Evaluar diferentes números de clústeres
    ks = range(2, 20)
    siluetas = []

    for k in ks:

        modelo = KMeans(
            n_clusters=k,
            max_iter=300,
            random_state=42,
            n_init=10
        )

        etiquetas = modelo.fit_predict(X)

        sil = metrics.silhouette_score(
            X,
            etiquetas
        )

        siluetas.append(sil)

    # Almacenar resultados
    resultados_silueta[escenario] = pd.DataFrame({
        'k': list(ks),
        'SILUETA': siluetas
    })

    # Identificar el mejor k
    mejor_indice = int(np.argmax(siluetas))
    mejor_k = list(ks)[mejor_indice]
    mejor_silueta = siluetas[mejor_indice]

    mejores_k[escenario] = mejor_k

    # Gráfico
    ax.plot(ks, siluetas, '-o')

    ax.scatter(
        mejor_k,
        mejor_silueta,
        s=85,
        zorder=3
    )

    ax.set_title(f'Escenario {escenario}')
    ax.set_xlabel('Número de clústeres (k)')
    ax.set_ylabel('Coeficiente de silueta')
    ax.set_xticks(list(ks))
    ax.grid(alpha=0.3)

    print(
        f'Escenario {escenario}: '
        f'mejor k = {mejor_k} | '
        f'silueta = {mejor_silueta:.4f}'
    )

plt.suptitle(
    'SNIES | Evaluación mediante silueta',
    fontsize=14
)

plt.tight_layout()
plt.show()

print('\nEvaluación mediante silueta finalizada.')
print('Las matrices originales permanecen intactas.')


In [ ]:

# ============================================================
# SNIES | CREACIÓN DE MODELOS DE CLUSTERING CON K-MEANS
# ESCENARIOS A, B Y C
# MATRICES NORMALIZADAS + VARIABLES DUMMY
#
# Entrena los modelos sin modificar las matrices originales.
# ============================================================

from sklearn.cluster import KMeans
from sklearn import set_config
from sklearn.metrics import silhouette_score
from IPython.display import display, Markdown
import pandas as pd
import numpy as np

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

set_config(display='diagram')

identificadores = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

columnas_excluir = identificadores + ['MODALIDAD']

valores_k = {
    'A': 8,
    'B': 5,
    'C': 2
}

matrices = {
    'A': matriz_A_modelo,
    'B': matriz_B_modelo,
    'C': matriz_C_modelo
}

caracteristicas_esperadas = {
    'A': 12,
    'B': 16,
    'C': 26
}

# ============================================================
# 2. ENTRENAMIENTO DE LOS MODELOS
# ============================================================

modelos = {}
caracteristicas_modelos = {}
resultados_modelos = []

for escenario, matriz in matrices.items():

    # Seleccionar características para K-means
    X = matriz.drop(
        columns=columnas_excluir
    )

    # Verificar dimensiones
    assert X.shape[1] == caracteristicas_esperadas[escenario], (
        f'Escenario {escenario}: '
        f'número inesperado de características.'
    )

    assert np.isfinite(
        X.to_numpy(dtype=float)
    ).all(), (
        f'Escenario {escenario}: '
        'se encontraron valores no finitos.'
    )

    # Número de clústeres seleccionado
    k = valores_k[escenario]

    # Crear y entrenar modelo K-means
    modelo = KMeans(
        n_clusters=k,
        max_iter=300,
        random_state=42,
        n_init=10
    )

    etiquetas = modelo.fit_predict(X)

    # Guardar modelo y nombres de características
    modelos[escenario] = modelo
    caracteristicas_modelos[escenario] = X.columns.tolist()

    # Calcular coeficiente de silueta
    silueta = silhouette_score(
        X,
        etiquetas
    )

    resultados_modelos.append({
        'ESCENARIO': escenario,
        'OFERTAS': len(X),
        'CARACTERISTICAS': X.shape[1],
        'K': k,
        'SILUETA': round(silueta, 4),
        'INERCIA_SSE': round(modelo.inertia_, 4)
    })

    # Mostrar representación visual del modelo
    display(
        Markdown(
            f'### Escenario {escenario} | k = {k}'
        )
    )

    display(modelo)

# ============================================================
# 3. RESUMEN DE LOS MODELOS
# ============================================================

resumen_modelos = pd.DataFrame(
    resultados_modelos
)

print("=" * 70)
print("SNIES | ENTRENAMIENTO DEFINITIVO K-MEANS")
print("=" * 70)

display(resumen_modelos)

print("\nModelos entrenados:", list(modelos.keys()))
print("Matrices originales: CONSERVADAS")
print("Matrices normalizadas: CONSERVADAS")
print("Matrices para modelado: SIN MODIFICAR")
print("Modelos disponibles para asignación de clústeres.")


# 3. Evaluación del Modelo


*   Inertia: valor pequeño esperado
*   Silueta: valor positivo esperado, idealmente mayor a 0.5



In [ ]:

# ============================================================
# SNIES | 3. EVALUACIÓN DEL MODELO K-MEANS
# INERCIA Y SILUETA - ESCENARIOS A, B Y C
# MATRICES NORMALIZADAS + VARIABLES DUMMY
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score
from IPython.display import display

# ============================================================
# 1. MATRICES UTILIZADAS EN EL ENTRENAMIENTO
# ============================================================

matrices = {
    'A': matriz_A_modelo,
    'B': matriz_B_modelo,
    'C': matriz_C_modelo
}

identificadores = [
    'CODIGO SNIES DEL PROGRAMA',
    'CODIGO DE LA INSTITUCION',
    'CODIGO DEL MUNICIPIO (PROGRAMA)',
    'ID MODALIDAD'
]

columnas_excluir = identificadores + ['MODALIDAD']

# ============================================================
# 2. EVALUACIÓN DE LOS MODELOS
# ============================================================

resultados = []

for escenario, matriz in matrices.items():

    # Recuperar modelo entrenado
    model = modelos[escenario]

    # Recuperar exactamente las características
    # utilizadas durante el entrenamiento
    X = matriz.drop(columns=columnas_excluir)

    # Comprobar nombres y orden de variables
    assert X.columns.tolist() == caracteristicas_modelos[escenario], (
        f'Escenario {escenario}: '
        'las características no coinciden con el entrenamiento.'
    )

    # Comprobar número de observaciones
    assert len(X) == len(model.labels_), (
        f'Escenario {escenario}: '
        'el número de observaciones no coincide.'
    )

    # Evaluar el modelo
    inercia = model.inertia_
    silueta = silhouette_score(
        X,
        model.labels_
    )

    resultados.append({
        'Escenario': escenario,
        'Ofertas': len(X),
        'Clústeres (k)': model.n_clusters,
        'Inercia (SSE)': inercia,
        'Silueta': silueta,
        '¿Silueta > 0.5?': (
            'Sí' if silueta > 0.5 else 'No'
        )
    })

# ============================================================
# 3. PRESENTACIÓN DE RESULTADOS
# ============================================================

evaluacion = pd.DataFrame(resultados)

display(
    evaluacion.style
    .format({
        'Inercia (SSE)': '{:.4f}',
        'Silueta': '{:.4f}'
    })
    .hide(axis='index')
    .set_caption(
        'SNIES | Evaluación de modelos K-means'
    )
)

print("\nEvaluación finalizada correctamente.")
print("Los modelos y las matrices permanecen intactos.")


# Los centroides no se calculan es parte del modelo de kmean dentro del modelo

# 4. Perfilamiento

Descripción de centroides

In [ ]:

# ============================================================
# SNIES | 4. PERFILAMIENTO
# CENTROIDES ALMACENADOS DENTRO DE K-MEANS
# ============================================================

from IPython.display import display, Markdown

for escenario in ['A', 'B', 'C']:
    display(Markdown(f"### Escenario {escenario} | Centroides"))
    display(modelos[escenario].cluster_centers_)


In [ ]:

# ============================================================
# SNIES | 4. PERFILAMIENTO
# CENTROIDES EN DATAFRAME DE PANDAS
# ESCENARIOS A, B Y C
#
# Recuperación directa desde los modelos K-means.
# No se recalculan los centroides.
# ============================================================

import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# 1. RECUPERAR CENTROIDES DE LOS MODELOS
# ============================================================

centroides_modelos = {}

for escenario in ['A', 'B', 'C']:

    # Modelo K-means entrenado
    modelo = modelos[escenario]

    # Nombres y orden exacto de las características
    variables_cluster = caracteristicas_modelos[escenario]

    # Recuperar los centroides almacenados por K-means
    centroides = pd.DataFrame(
        modelo.cluster_centers_,
        columns=variables_cluster
    )

    # Identificar cada clúster
    centroides.index.name = 'Clúster'

    # Guardar resultados para consultas posteriores
    centroides_modelos[escenario] = centroides

    # Presentación
    display(
        Markdown(
            f'### Escenario {escenario} | '
            f'Centroides de K-means'
        )
    )

    display(centroides.round(4))

# ============================================================
# 2. VERIFICACIÓN
# ============================================================

assert centroides_modelos['A'].shape == (8, 12)
assert centroides_modelos['B'].shape == (5, 16)
assert centroides_modelos['C'].shape == (2, 26)

print("=" * 65)
print("CENTROIDES RECUPERADOS CORRECTAMENTE")
print("=" * 65)

print("Escenario A:", centroides_modelos['A'].shape)
print("Escenario B:", centroides_modelos['B'].shape)
print("Escenario C:", centroides_modelos['C'].shape)

print("\nNo se recalcularon centroides.")
print("Los modelos originales permanecen intactos.")


In [ ]:

# ============================================================
# SNIES | 4. PERFILAMIENTO
# DESNORMALIZACIÓN DE CENTROIDES
# ESCENARIOS A, B Y C
#
# Los centroides se recuperan del modelo K-means.
# No se recalculan.
# Las matrices y modelos permanecen intactos.
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

centroides_desnormalizados = {}

dimensiones_esperadas = {
    'A': (8, 12),
    'B': (5, 16),
    'C': (2, 26)
}

# ============================================================
# 2. RECUPERAR Y DESNORMALIZAR CENTROIDES
# ============================================================

for escenario in ['A', 'B', 'C']:

    # Recuperar modelo y escalador correspondientes
    modelo = modelos[escenario]
    scaler = escaladores[escenario]

    # Recuperar nombres y orden exactos del entrenamiento
    variables_cluster = caracteristicas_modelos[escenario]

    # Extraer los centroides almacenados por K-means
    centroides = pd.DataFrame(
        modelo.cluster_centers_,
        columns=variables_cluster
    )

    # Identificar las variables numéricas normalizadas
    variables_a_desnormalizar = variables_numericas[escenario]

    # Verificar correspondencia con el escalador
    assert list(scaler.feature_names_in_) == (
        variables_a_desnormalizar
    ), (
        f'Escenario {escenario}: '
        'el escalador no corresponde a las variables.'
    )

    assert all(
        columna in centroides.columns
        for columna in variables_a_desnormalizar
    ), (
        f'Escenario {escenario}: '
        'faltan variables para desnormalizar.'
    )

    # Recuperar las unidades originales
    # SIN recalcular centroides
    centroides[variables_a_desnormalizar] = (
        scaler.inverse_transform(
            centroides[variables_a_desnormalizar]
        )
    )

    # Identificar clústeres
    centroides.index = pd.Index(
        range(modelo.n_clusters),
        name='Clúster'
    )

    # Validaciones
    assert centroides.shape == dimensiones_esperadas[escenario]

    assert np.isfinite(
        centroides.to_numpy(dtype=float)
    ).all(), (
        f'Escenario {escenario}: '
        'centroides con valores no finitos.'
    )

    # Conservar resultados
    centroides_desnormalizados[escenario] = centroides

    # Presentación
    display(
        Markdown(
            f'### Escenario {escenario} | '
            'Centroides desnormalizados'
        )
    )

    display(centroides.round(2))

# ============================================================
# 3. VERIFICACIÓN FINAL
# ============================================================

print("=" * 65)
print("SNIES | CENTROIDES DESNORMALIZADOS")
print("=" * 65)

for escenario, centroides in (
    centroides_desnormalizados.items()
):

    print(
        f'Escenario {escenario}: '
        f'{centroides.shape[0]} centroides | '
        f'{centroides.shape[1]} características'
    )

print("\nCentroides recuperados de K-means.")
print("Variables cuantitativas en unidades originales.")
print("Variables dummy conservadas.")
print("No se recalcularon los centroides.")
print("Los modelos y las matrices permanecen intactos.")



# 4. Perfilamiento de los clústeres — SNIES 2020–2025

## 4.1. Criterios de interpretación

El perfilamiento consiste en describir las características de las ofertas académicas agrupadas mediante K-means, a partir de los centroides desnormalizados. Cada centroide representa el comportamiento promedio de las ofertas pertenecientes a un clúster.

Para interpretar los resultados se consideran los siguientes indicadores:

- **Volumen de demanda:** número de inscritos en cada semestre.
- **Matrícula de primer curso:** número de matriculados de primer curso registrados durante el período.
- **Proporción de admisión:** admitidos / inscritos.
- **Proporción de ingreso de admitidos:** matriculados de primer curso / admitidos.
- **Ingreso relativo:** matriculados de primer curso / inscritos.
- **Tendencia histórica:** pendiente estimada mediante regresión lineal para inscritos o matriculados de primer curso, utilizando los años completos disponibles de cada oferta entre 2020 y 2025, con un mínimo de cuatro años. Se expresa en estudiantes por año.

### Definición e interpretación del ingreso relativo

En este estudio, **ingreso relativo** corresponde al cociente entre los matriculados de primer curso y los inscritos de una oferta académica.

**Ingreso relativo = Matriculados de primer curso / Inscritos**

Por ejemplo, un ingreso relativo de 0.80 representa una relación agregada de aproximadamente 80 matriculados de primer curso por cada 100 inscritos.

Este indicador no debe confundirse con la proporción de admisión ni con la proporción de ingreso de admitidos.

Debido a que algunos registros SNIES presentan proporciones superiores a 1, el ingreso relativo se interpreta como un cociente agregado entre cantidades registradas y no como una probabilidad individual de matrícula.

### Interpretación de los centroides y desnormalización

Los centroides **no se calculan nuevamente** durante el perfilamiento: se recuperan directamente del atributo `cluster_centers_` de cada modelo K-means entrenado.

Como los modelos utilizaron indicadores cuantitativos escalados con `MinMaxScaler`, se aplica `inverse_transform()` **únicamente a las coordenadas cuantitativas**, utilizando el escalador específico de cada escenario (A, B o C).

Esta operación recupera las unidades originales —estudiantes, estudiantes por año y cocientes— sin alterar los modelos ni volver a calcular los centroides.

Los valores de las proporciones presentados en los centroides corresponden al promedio de los indicadores calculados previamente para cada oferta académica. En consecuencia, **el promedio de las proporciones no necesariamente coincide con la proporción entre los promedios** de inscritos y matriculados del clúster.

### Interpretación de las variables institucionales dummy

Las variables institucionales `SECTOR IES_Privado` y `TIPO IES_Seccional` fueron codificadas como variables binarias (0/1) mediante `pd.get_dummies(..., drop_first=True)`.

Su coordenada en el centroide representa la **fracción de ofertas del clúster** que pertenecen a la categoría indicada.

Por ejemplo, un valor de 0.29 en `TIPO IES_Seccional` indica que aproximadamente el **29 % de las ofertas del grupo** son seccionales; no significa que una oferta sea «29 % seccional».

Los valores 1 y 0 indican, respectivamente, totalidad y ausencia de esa categoría en el grupo.

Estas variables dummy **no se desnormalizan**, dado que no pasaron por `MinMaxScaler` y ya poseen una interpretación directa. Tampoco se redondean a 0 o 1 para describir un grupo mixto, pues ello eliminaría información sobre su composición.

La variable `MODALIDAD` se conserva como atributo contextual y **no participa como característica de entrada** de K-means.

La composición institucional complementa el perfilamiento académico, pero no demuestra por sí misma efectos causales del sector o del tipo de institución sobre la demanda.

Los términos **demanda alta, intermedia o baja** e **ingreso relativo alto, moderado o bajo** se utilizan comparativamente entre los centroides de un mismo escenario. No corresponden a umbrales institucionales previamente establecidos.

Las pendientes históricas corresponden a variaciones absolutas estimadas en estudiantes por año y no a porcentajes de crecimiento. Una pendiente próxima a cero representa una variación absoluta pequeña, pero no demuestra por sí sola estabilidad estadística.

---

## 4.2. Escenario A — Perfiles de demanda académica en 2025

**Ofertas analizadas:** 124

**Número de clústeres:** 8

**Coeficiente de silueta:** 0.5300

Este escenario caracteriza las ofertas académicas según el volumen de inscritos, la matrícula de primer curso, las proporciones de admisión e ingreso y las características institucionales registradas durante 2025.

| Clúster | Ofertas | Perfil académico | Descripción |
|---|---:|---|---|
| 0 | 50 | Demanda baja con ingreso relativo alto | Promedio de 79 inscritos en S1 y 43 en S2. Ingreso relativo de 0.81 en ambos semestres. Sector oficial, tipo principal. |
| 1 | 5 | Demanda baja con ingreso relativo moderado | Promedio de 88 inscritos en S1 y 31 en S2. Ingreso relativo de 0.47 y 0.48. Sector privado, tipo seccional. |
| 2 | 28 | Demanda muy baja con ingreso relativo favorable | Promedio de 42 inscritos en S1 y 23 en S2. Ingreso relativo de 0.69 y 0.76. Sector privado, tipo principal. |
| 3 | 9 | Demanda alta con ingreso relativo bajo | Promedio de 303 inscritos en S1 y 182 en S2. Ingreso relativo de 0.18 y 0.25. Sector oficial, tipo principal. |
| 4 | 6 | Demanda intermedia-alta con ingreso relativo bajo | Promedio de 196 inscritos en S1 y 143 en S2. Ingreso relativo de 0.23 y 0.30. Sector oficial, tipo seccional. |
| 5 | 8 | Demanda alta con elevada matrícula de primer curso | Promedio de 361 inscritos en S1 y 210 en S2, con aproximadamente 134 matriculados de primer curso en cada semestre. Sector oficial, tipo principal. |
| 6 | 14 | Demanda baja con ingreso relativo muy alto | Promedio de 59 inscritos en S1 y 32 en S2. Ingreso relativo de 0.85 y 0.90. Sector privado, tipo seccional. |
| 7 | 4 | Demanda intermedia con ingreso relativo alto | Promedio de 149 inscritos en S1 y 80 en S2. Ingreso relativo de 0.80 y 0.81. Sector oficial, tipo seccional. |

**Interpretación:** el escenario A permite diferenciar ofertas de alta, intermedia y baja demanda, así como distintos niveles de ingreso relativo. Destaca el clúster 5 por su elevada matrícula de primer curso, mientras que los clústeres 3 y 4 presentan volúmenes importantes de inscritos, pero proporciones relativamente bajas de ingreso.

Este escenario describe la situación registrada en 2025 y **no permite establecer por sí solo tendencias históricas de crecimiento o disminución**.

---

## 4.3. Escenario B — Perfiles de evolución histórica 2020–2025

**Ofertas analizadas:** 115

**Número de clústeres:** 5

**Coeficiente de silueta:** 0.4452

Este escenario incorpora promedios históricos, proporciones de admisión e ingreso y pendientes anuales de inscritos y matriculados de primer curso, diferenciadas por semestre.

Las pendientes se estimaron utilizando exclusivamente los años con información completa en ambos semestres para las variables centrales del análisis. Cada oferta debía disponer de al menos cuatro años completos dentro del período 2020–2025.

| Clúster | Ofertas | Perfil académico | Descripción |
|---|---:|---|---|
| 0 | 10 | Demanda intermedia con crecimiento del ingreso efectivo | Pendientes de inscritos de +2.38 en S1 y +7.94 en S2. Las pendientes de matriculados de primer curso son +6.57 y +3.48 estudiantes/año. Sector oficial, tipo seccional. |
| 1 | 44 | Demanda baja con estabilidad relativa de inscritos | Pendientes de inscritos de +0.21 en S1 y −1.97 en S2. Las pendientes de matriculados de primer curso son +3.32 y +0.81 estudiantes/año. Sector oficial, tipo principal. |
| 2 | 14 | Demanda baja con crecimiento moderado | Pendientes de inscritos de +3.79 en S1 y +2.45 en S2. Las pendientes de matriculados de primer curso son +1.20 y +1.58 estudiantes/año. Sector privado, tipo seccional. |
| 3 | 29 | Demanda reducida con variaciones históricas pequeñas | Pendientes de inscritos de +0.60 en S1 y +0.68 en S2. Las pendientes de matriculados de primer curso son +0.52 y +1.18 estudiantes/año. Sector privado, tipo principal. |
| 4 | 18 | Demanda alta con crecimiento de inscritos superior al de nuevos matriculados | Pendientes de inscritos de +18.00 en S1 y +10.87 en S2, frente a pendientes de matriculados de primer curso de +0.94 y +1.21 estudiantes/año. Sector oficial, tipo principal. |

**Interpretación:** el clúster 4 presenta el mayor crecimiento absoluto promedio de inscritos, sin un incremento equivalente de matriculados de primer curso. El clúster 0 evidencia incrementos tanto en inscritos como en matriculados de primer curso, mientras que el clúster 1 presenta una pendiente promedio negativa de inscritos durante el segundo semestre.

En el perfil del clúster 0, la expresión **ingreso efectivo** se refiere al número de matriculados de primer curso y no a la proporción de ingreso relativo.

Las pendientes representan cambios absolutos estimados en estudiantes por año. No deben confundirse con tasas porcentuales de crecimiento ni interpretarse automáticamente como cambios estadísticamente significativos.

**Limitación:** no puede afirmarse que todas las ofertas de un mismo clúster crezcan, disminuyan o permanezcan estables. Los centroides representan comportamientos promedio que pueden ocultar trayectorias individuales diferentes.

---

## 4.4. Escenario C — Perfiles integrales de demanda

**Ofertas analizadas:** 105

**Número de clústeres:** 2

**Coeficiente de silueta:** 0.3973

Este escenario combina las características históricas del período 2020–2025 con el comportamiento observado en 2025, permitiendo una caracterización integral de las ofertas académicas.

| Clúster | Ofertas | Perfil académico | Descripción |
|---|---:|---|---|
| 0 | 24 | Demanda elevada con crecimiento histórico de inscritos | Promedios históricos de 275 inscritos en S1 y 165 en S2. Pendientes de +13.88 y +11.23 estudiantes/año. En 2025 registra promedios de 289 inscritos en S1 y 174 en S2. |
| 1 | 81 | Demanda reducida con relativa estabilidad histórica de inscritos | Promedios históricos de 71 inscritos en S1 y 44 en S2. Pendientes de +1.04 y −0.40 estudiantes/año. En 2025 registra promedios de 71 inscritos en S1 y 39 en S2. |

**Interpretación:** el escenario C distingue dos grandes perfiles de demanda. El primero agrupa ofertas con mayor volumen de inscritos y crecimiento histórico positivo en ambos semestres; el segundo concentra ofertas de menor volumen y pendientes históricas promedio relativamente próximas a cero.

En cuanto a su composición institucional, el clúster 0 presenta un centroide de 0.00 en `SECTOR IES_Privado` y aproximadamente 0.29 en `TIPO IES_Seccional`. Esto indica que las ofertas del grupo pertenecen al sector oficial y que aproximadamente el 29 % corresponde al tipo seccional.

Por su parte, el clúster 1 registra aproximadamente 0.44 en `SECTOR IES_Privado` y 0.20 en `TIPO IES_Seccional`, lo que refleja una composición institucional mixta.

El coeficiente de silueta de 0.3973 evidencia una estructura de agrupación con separación limitada entre los dos clústeres. Este valor debe interpretarse dentro de las características específicas del escenario, sin establecer una comparación directa de calidad con los demás escenarios, dado que utilizan diferentes características y conjuntos de ofertas.

La denominación **relativa estabilidad histórica** describe exclusivamente el comportamiento promedio del centroide del clúster 1; no implica que todas sus ofertas presenten trayectorias individuales estables.

---

## 4.5. Interpretación conjunta de los escenarios

| Escenario | Enfoque principal | Contribución al análisis |
|---|---|---|
| A | Demanda reciente de 2025 | Caracterizar el volumen de inscritos, la matrícula de primer curso y el ingreso relativo. |
| B | Evolución histórica 2020–2025 | Identificar patrones promedio de crecimiento y estabilidad relativa de inscritos y matriculados de primer curso. |
| C | Demanda integral | Relacionar la trayectoria histórica de las ofertas con su situación en 2025. |

Los tres escenarios aportan perspectivas complementarias para caracterizar la demanda académica.

El **escenario A** permite diagnosticar la situación reciente, diferenciando ofertas por volumen de demanda e ingreso relativo.

El **escenario B** incorpora la dimensión temporal, facilitando la identificación de patrones históricos mediante promedios y pendientes diferenciadas por semestre.

El **escenario C** integra ambas perspectivas y permite reconocer perfiles generales que relacionan el comportamiento reciente con la evolución histórica.

### Limitaciones del perfilamiento

Las denominaciones de los perfiles se fundamentan en los centroides desnormalizados, que representan características promedio de las ofertas agrupadas.

Por tanto:

- Los valores de los centroides no describen necesariamente a cada oferta individual.
- Las proporciones promedio no deben confundirse con cocientes calculados a partir de los promedios de otras variables.
- Las pendientes representan cambios absolutos estimados, no tasas porcentuales ni pruebas de significancia estadística.
- Los coeficientes de silueta deben interpretarse considerando las particularidades de cada escenario.
- Las características institucionales representadas mediante dummies permiten describir la composición de los grupos, pero no determinan por sí solas la trayectoria académica de las ofertas.
- La variable modalidad se conserva como atributo contextual y no interviene directamente en el cálculo de distancias de K-means.

Para identificar programas académicos específicos con demanda creciente, estable o decreciente es necesario complementar el perfilamiento con las pendientes individuales de cada oferta y relacionarlas con el nombre del programa, la institución, el municipio y la modalidad.

En consecuencia, los perfiles obtenidos mediante K-means constituyen una **caracterización estadística exploratoria de la demanda académica**, y no una clasificación definitiva del comportamiento individual de cada programa.


In [ ]:

# ============================================================
# SNIES ANALYTICS 2020-2025
# ASIGNACIÓN DE CLÚSTERES Y EXPORTACIÓN DEFINITIVA
# ESCENARIOS A, B Y C
#
# No reentrena modelos ni recalcula centroides.
# Conserva los indicadores en sus unidades originales.
# Ejecución repetible, sin columnas CLUSTER duplicadas.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

base = Path(
    "/content/drive/MyDrive/MaestriaUPB_Mineria_Datos/P_final"
)
destino = base / "streamlit"
destino.mkdir(parents=True, exist_ok=True)

claves = [
    "CODIGO SNIES DEL PROGRAMA",
    "CODIGO DE LA INSTITUCION",
    "CODIGO DEL MUNICIPIO (PROGRAMA)",
    "ID MODALIDAD"
]

contexto_cols = [
    "NOMBRE_ESTANDARIZADO",
    "INSTITUCION DE EDUCACION SUPERIOR (IES)",
    "MUNICIPIO_ESTANDARIZADO",
    "SECTOR IES",
    "TIPO IES",
    "MODALIDAD"
]

descriptivas = [
    "PROGRAMA ACADEMICO",
    "INSTITUCION DE EDUCACION SUPERIOR (IES)",
    "MUNICIPIO DE OFERTA DEL PROGRAMA"
]

dummies = ["SECTOR IES_Privado", "TIPO IES_Seccional"]

matrices = {"A": matriz_A, "B": matriz_B, "C": matriz_C}
matrices_modelo = {
    "A": matriz_A_modelo,
    "B": matriz_B_modelo,
    "C": matriz_C_modelo
}

esperado = {
    "A": (124, 8, 24),
    "B": (115, 5, 28),
    "C": (105, 2, 38)
}

assert df.shape == (3896, 60)
assert df[claves].drop_duplicates().shape[0] == 243

# Catálogo y contexto de la observación más reciente
catalogo = (
    df.sort_values(["ANO", "SEMESTRE"])
      .drop_duplicates(claves, keep="last")
)

assert not catalogo.duplicated(claves).any()

# ============================================================
# 2. ASIGNACIÓN, IDENTIFICACIÓN Y CENTROIDES
# ============================================================

matrices_etiquetadas = {}
resultados = {}
centroides_exportacion = {}
programas_por_cluster = {}

for escenario in "ABC":

    n, k, n_columnas = esperado[escenario]
    modelo = modelos[escenario]
    scaler = escaladores[escenario]

    # Matriz original: no contiene indicadores normalizados
    matriz = matrices[escenario].drop(
        columns="CLUSTER", errors="ignore"
    ).copy()

    entrenamiento = matrices_modelo[escenario]

    assert len(matriz) == len(entrenamiento) == n
    assert modelo.n_clusters == k
    assert len(modelo.labels_) == n
    assert not matriz.duplicated(claves).any()

    # Comprobar el orden de las ofertas del entrenamiento
    assert matriz[claves].reset_index(drop=True).equals(
        entrenamiento[claves].reset_index(drop=True)
    ), f"Escenario {escenario}: cambió el orden de las ofertas."

    # Asignar etiquetas directamente desde K-means
    matriz["CLUSTER"] = np.asarray(modelo.labels_, dtype=int)
    assert matriz["CLUSTER"].nunique() == k

    matrices_etiquetadas[escenario] = matriz

    # Identificar programas por clúster
    programas = (
        matriz[claves + ["CLUSTER"]]
        .merge(
            catalogo[claves + descriptivas],
            on=claves, how="left", validate="one_to_one"
        )
        [["CLUSTER"] + descriptivas + claves]
        .sort_values(["CLUSTER", "PROGRAMA ACADEMICO"])
        .reset_index(drop=True)
    )

    assert programas[descriptivas].notna().all().all()
    programas_por_cluster[escenario] = programas

    # Recuperar centroides del modelo y desnormalizar
    # únicamente las características cuantitativas
    variables = caracteristicas_modelos[escenario]
    numericas = variables_numericas[escenario]

    assert list(scaler.feature_names_in_) == numericas
    assert len(variables) == modelo.cluster_centers_.shape[1]

    centroides = pd.DataFrame(
        modelo.cluster_centers_,
        columns=variables,
        index=pd.Index(range(k), name="Clúster")
    )

    centroides[numericas] = scaler.inverse_transform(
        centroides[numericas]
    )

    centroides_exportacion[escenario] = centroides

    # Datos para Streamlit:
    # indicadores originales + CLUSTER + dos dummies + contexto
    tabla = matriz.drop(
        columns=["SECTOR IES", "TIPO IES", "MODALIDAD"]
    ).copy()

    # Las dummies se recuperan del entrenamiento,
    # respetando exactamente el orden validado
    for columna in dummies:
        assert columna in entrenamiento.columns
        tabla[columna] = entrenamiento[columna].to_numpy(dtype=int)

    tabla["ESCENARIO"] = escenario

    tabla = tabla.merge(
        catalogo[claves + contexto_cols],
        on=claves,
        how="left",
        validate="one_to_one"
    )

    assert tabla.shape == (n, n_columnas), (
        f"Escenario {escenario}: dimensiones {tabla.shape}; "
        f"esperadas {(n, n_columnas)}"
    )
    assert tabla["CLUSTER"].notna().all()
    assert not tabla.duplicated(claves).any()
    assert not any(
        c in tabla.columns
        for c in ["CLUSTER_x", "CLUSTER_y"]
    )
    assert tabla[contexto_cols].notna().all().all()

    resultados[escenario] = tabla

    display(Markdown(f"### Escenario {escenario} | Ofertas por clúster"))
    display(
        matriz["CLUSTER"].value_counts().sort_index()
        .rename_axis("Clúster")
        .reset_index(name="Número de ofertas")
    )

    display(Markdown(f"### Escenario {escenario} | Programas académicos"))
    display(programas)

# ============================================================
# 3. CONSOLIDACIÓN
# ============================================================

resultados_consolidados = pd.concat(
    [resultados[e] for e in "ABC"],
    ignore_index=True
)

assert len(resultados_consolidados) == 344
assert not resultados_consolidados.duplicated(
    claves + ["ESCENARIO"]
).any()

# ============================================================
# 4. EXPORTACIÓN A EXCEL Y CSV
# ============================================================

ruta_excel = base / "Resultados_Clustering_SNIES.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    for escenario in "ABC":
        matrices_etiquetadas[escenario].to_excel(
            writer, sheet_name=f"Ofertas_{escenario}", index=False
        )
        centroides_exportacion[escenario].to_excel(
            writer, sheet_name=f"Centroides_{escenario}"
        )

rutas_csv = {}

for escenario in "ABC":
    ruta_csv = destino / f"resultados_cluster_{escenario}.csv"
    resultados[escenario].to_csv(
        ruta_csv, index=False, encoding="utf-8-sig"
    )
    rutas_csv[escenario] = ruta_csv

ruta_consolidado = destino / "resultados_cluster_consolidados.csv"
resultados_consolidados.to_csv(
    ruta_consolidado, index=False, encoding="utf-8-sig"
)

# ============================================================
# 5. VERIFICACIÓN DE ARCHIVOS EXPORTADOS
# ============================================================

assert ruta_excel.is_file()
assert ruta_consolidado.is_file()

for escenario in "ABC":
    n, k, columnas = esperado[escenario]

    archivo = pd.read_csv(
        rutas_csv[escenario],
        encoding="utf-8-sig",
        dtype={c: str for c in claves},
        low_memory=False
    )

    assert archivo.shape == (n, columnas)
    assert archivo["CLUSTER"].nunique() == k
    assert np.array_equal(
        archivo["CLUSTER"].to_numpy(dtype=int),
        modelos[escenario].labels_
    )

# Actualizar matrices originales solo después
# de completar las validaciones y exportaciones
matriz_A = matrices_etiquetadas["A"]
matriz_B = matrices_etiquetadas["B"]
matriz_C = matrices_etiquetadas["C"]

# ============================================================
# 6. REPORTE FINAL
# ============================================================

print("\n" + "=" * 70)
print("SNIES | EXPORTACIÓN DEFINITIVA VALIDADA")
print("=" * 70)

display(pd.DataFrame([
    {
        "ESCENARIO": e,
        "OFERTAS": len(resultados[e]),
        "CLÚSTERES": resultados[e]["CLUSTER"].nunique(),
        "COLUMNAS CSV": resultados[e].shape[1],
        "FALTANTES CLUSTER": resultados[e]["CLUSTER"].isna().sum()
    }
    for e in "ABC"
]))

print("\nDISTRIBUCIÓN DE OFERTAS POR CLÚSTER")
display(
    resultados_consolidados
    .groupby(["ESCENARIO", "CLUSTER"])
    .size()
    .to_frame("OFERTAS")
)

print("\nARCHIVO EXCEL:", ruta_excel)
print("\nARCHIVOS CSV:")
for escenario in "ABC":
    print(rutas_csv[escenario])
print(ruta_consolidado)

print("\nModelos K-means: SIN REENTRENAMIENTO")
print("Centroides: RECUPERADOS DEL MODELO")
print("Indicadores académicos: EN UNIDADES ORIGINALES")
print("Matrices etiquetadas: A, B y C")
print("Exportación finalizada correctamente.")


In [ ]:

# ============================================================
# SNIES ANALYTICS 2020-2025
# CELDA 2 | CREACIÓN DE LA APLICACIÓN STREAMLIT
#
# - 3 escenarios: A, B y C
# - 15 perfiles y descripciones académicas
# - 4 pestañas interactivas
# - Fondo institucional UPB
# - Validación de CSV
# - Respaldo automático de app.py
#
# No modifica modelos, matrices, centroides ni CSV.
# ============================================================

from pathlib import Path
from datetime import datetime
import ast
import pandas as pd

carpeta = Path(
    "/content/drive/MyDrive/"
    "MaestriaUPB_Mineria_Datos/P_final/streamlit"
)

carpeta.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. CÓDIGO DE LA APLICACIÓN
# ============================================================

codigo_app = r'''
import base64
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

st.set_page_config(
    page_title="SNIES Analytics 2020–2025",
    page_icon="🎓",
    layout="wide"
)

BASE = Path(__file__).resolve().parent

CLAVES = [
    "CODIGO SNIES DEL PROGRAMA",
    "CODIGO DE LA INSTITUCION",
    "CODIGO DEL MUNICIPIO (PROGRAMA)",
    "ID MODALIDAD"
]

PROGRAMA = "NOMBRE_ESTANDARIZADO"
IES = "INSTITUCION DE EDUCACION SUPERIOR (IES)"
MUNICIPIO = "MUNICIPIO_ESTANDARIZADO"

DUMMIES = [
    "SECTOR IES_Privado",
    "TIPO IES_Seccional"
]

ESCENARIOS = {
    "A": {
        "nombre": "Demanda reciente 2025",
        "n": 124,
        "k": 8,
        "columnas": 24,
        "silueta": 0.5300
    },
    "B": {
        "nombre": "Evolución histórica 2020–2025",
        "n": 115,
        "k": 5,
        "columnas": 28,
        "silueta": 0.4452
    },
    "C": {
        "nombre": "Demanda integral",
        "n": 105,
        "k": 2,
        "columnas": 38,
        "silueta": 0.3973
    }
}

# ============================================================
# 2. NOMBRES DE LOS 15 PERFILES ACADÉMICOS
# ============================================================

PERFILES = {
    "A": {
        0: "Demanda baja con ingreso relativo alto",
        1: "Demanda baja con ingreso relativo moderado",
        2: "Demanda muy baja con ingreso relativo favorable",
        3: "Demanda alta con ingreso relativo bajo",
        4: "Demanda intermedia-alta con ingreso relativo bajo",
        5: "Demanda alta con elevada matrícula de primer curso",
        6: "Demanda baja con ingreso relativo muy alto",
        7: "Demanda intermedia con ingreso relativo alto"
    },
    "B": {
        0: "Demanda intermedia con crecimiento del ingreso efectivo",
        1: "Demanda baja con estabilidad relativa de inscritos",
        2: "Demanda baja con crecimiento moderado",
        3: "Demanda reducida con variaciones históricas pequeñas",
        4: "Demanda alta con crecimiento de inscritos superior al de nuevos matriculados"
    },
    "C": {
        0: "Demanda elevada con crecimiento histórico de inscritos",
        1: "Demanda reducida con relativa estabilidad histórica de inscritos"
    }
}


# ============================================================
# 3. INTERPRETACIÓN ACADÉMICA DE LOS 15 CLÚSTERES
# SIGNIFICADO DE LOS PERFILES OBTENIDOS MEDIANTE K-MEANS
#
# Las denominaciones son comparativas dentro de cada
# escenario; no corresponden a umbrales oficiales del SNIES.
# ============================================================

DESCRIPCIONES = {
    "A": {
        0: (
            "Agrupa ofertas académicas que registran un volumen "
            "relativamente reducido de inscritos, pero presentan "
            "una relación elevada entre los matriculados de "
            "primer curso y los inscritos. Esto significa que, "
            "aunque su demanda es baja, el ingreso relativo "
            "es favorable frente a otros grupos del escenario."
        ),
        1: (
            "Agrupa ofertas con un número relativamente bajo "
            "de inscritos y una relación intermedia entre "
            "matriculados de primer curso e inscritos. "
            "Se caracteriza por una demanda reducida y un "
            "ingreso relativo moderado en comparación con "
            "los demás clústeres."
        ),
        2: (
            "Identifica ofertas con uno de los menores volúmenes "
            "de inscritos del escenario. Sin embargo, presentan "
            "una relación relativamente favorable entre "
            "matriculados de primer curso e inscritos. "
            "El perfil combina una demanda muy reducida con "
            "un ingreso relativo comparativamente favorable."
        ),
        3: (
            "Agrupa ofertas que atraen un volumen elevado de "
            "inscritos, pero registran una relación relativamente "
            "baja entre matriculados de primer curso e inscritos. "
            "Esto indica que una demanda alta no necesariamente "
            "se acompaña de un ingreso relativo igualmente alto."
        ),
        4: (
            "Identifica ofertas con un volumen de inscritos "
            "intermedio-alto respecto a los demás grupos, "
            "pero con una relación reducida entre matriculados "
            "de primer curso e inscritos. El perfil combina "
            "una demanda considerable con un ingreso relativo "
            "comparativamente bajo."
        ),
        5: (
            "Agrupa ofertas con una demanda académica elevada "
            "y un volumen destacado de matriculados de primer "
            "curso. Se diferencia de los demás perfiles porque "
            "presenta simultáneamente numerosos inscritos y "
            "una matrícula de primer curso comparativamente "
            "alta en términos absolutos."
        ),
        6: (
            "Agrupa ofertas académicas con un volumen "
            "relativamente reducido de inscritos, pero con una "
            "relación muy elevada entre matriculados de primer "
            "curso e inscritos. Su característica distintiva "
            "no es atraer muchos inscritos, sino presentar "
            "uno de los ingresos relativos más altos del escenario."
        ),
        7: (
            "Agrupa ofertas con un volumen de inscritos "
            "intermedio y una relación elevada entre "
            "matriculados de primer curso e inscritos. "
            "Representa un perfil que combina una demanda "
            "moderada con un ingreso relativo favorable "
            "frente a otros grupos."
        )
    },

    "B": {
        0: (
            "Agrupa ofertas con un volumen histórico de inscritos "
            "intermedio y una tendencia promedio creciente en "
            "los matriculados de primer curso. El perfil "
            "destaca por el aumento absoluto estimado de nuevos "
            "matriculados, aunque los inscritos también presentan "
            "variaciones positivas de distinta magnitud."
        ),
        1: (
            "Agrupa ofertas con una demanda histórica "
            "relativamente baja y pendientes promedio de "
            "inscritos cercanas a cero, con diferencias entre "
            "semestres. Esto describe variaciones absolutas "
            "reducidas del grupo, sin demostrar que todas las "
            "ofertas mantengan una demanda estable."
        ),
        2: (
            "Identifica ofertas con un volumen histórico "
            "reducido de inscritos y pendientes promedio "
            "positivas, pero de magnitud moderada. El perfil "
            "refleja una tendencia creciente en términos "
            "absolutos sin alcanzar los aumentos observados "
            "en los grupos de mayor demanda."
        ),
        3: (
            "Agrupa ofertas con un volumen histórico de "
            "inscritos reducido y pendientes promedio "
            "de pequeña magnitud tanto en inscritos como "
            "en matriculados de primer curso. Describe "
            "cambios históricos absolutos limitados, "
            "sin implicar ausencia de fluctuaciones "
            "en las ofertas individuales."
        ),
        4: (
            "Agrupa ofertas con demanda histórica elevada "
            "y pendientes positivas de inscritos "
            "considerablemente mayores que las pendientes "
            "de matriculados de primer curso. Esto significa "
            "que el número de inscritos presenta un aumento "
            "absoluto estimado superior al de los nuevos "
            "matriculados, sin establecer por sí mismo "
            "las causas de esa diferencia."
        )
    },

    "C": {
        0: (
            "Agrupa ofertas con un volumen de inscritos "
            "comparativamente elevado, tanto en su comportamiento "
            "histórico como en la situación reciente. "
            "Sus pendientes históricas promedio son positivas "
            "y de mayor magnitud que las del otro grupo. "
            "El perfil combina alta demanda con una tendencia "
            "histórica creciente de inscritos."
        ),
        1: (
            "Agrupa ofertas con un volumen de inscritos "
            "comparativamente reducido y pendientes históricas "
            "promedio de pequeña magnitud, próximas a cero. "
            "El perfil describe una demanda menor con cambios "
            "históricos absolutos limitados en términos "
            "promedio, sin demostrar estabilidad estadística "
            "ni uniformidad entre las ofertas."
        )
    }
}


# ============================================================
# 4. INDICADORES ACADÉMICOS
# ============================================================

INDICADORES = {
    "A": {
        "Inscritos S1 — 2025": "INSCRITOS_2025_S1",
        "Inscritos S2 — 2025": "INSCRITOS_2025_S2",
        "Matrícula primer curso S1 — 2025":
            "MATRICULADOS PRIMER CURSO_2025_S1",
        "Matrícula primer curso S2 — 2025":
            "MATRICULADOS PRIMER CURSO_2025_S2",
        "Admisión S1 — 2025": "PROP_ADMISION_2025_S1",
        "Admisión S2 — 2025": "PROP_ADMISION_2025_S2",
        "Ingreso de admitidos S1 — 2025":
            "PROP_INGRESO_ADMITIDOS_2025_S1",
        "Ingreso de admitidos S2 — 2025":
            "PROP_INGRESO_ADMITIDOS_2025_S2",
        "Ingreso relativo S1 — 2025":
            "PROP_INGRESO_INSCRITOS_2025_S1",
        "Ingreso relativo S2 — 2025":
            "PROP_INGRESO_INSCRITOS_2025_S2"
    },
    "B": {},
    "C": {}
}

for s in (1, 2):

    for variable, etiqueta in [
        ("INSCRITOS", "Inscritos"),
        ("MATRICULADOS PRIMER CURSO", "Matrícula primer curso")
    ]:

        INDICADORES["B"][
            f"Promedio histórico {etiqueta} S{s}"
        ] = f"{variable}_PROM_S{s}"

        INDICADORES["B"][
            f"Pendiente {etiqueta} S{s} (estudiantes/año)"
        ] = f"{variable}_PENDIENTE_S{s}"

        INDICADORES["C"][
            f"Promedio histórico {etiqueta} S{s}"
        ] = f"{variable}_PROM_S{s}"

        INDICADORES["C"][
            f"Pendiente {etiqueta} S{s} (estudiantes/año)"
        ] = f"{variable}_PENDIENTE_S{s}"

        INDICADORES["C"][
            f"{etiqueta} S{s} — 2025"
        ] = f"{variable}_2025_S{s}"

    for sufijo, titulo in [
        ("ADMISION", "Admisión"),
        ("INGRESO_ADMITIDOS", "Ingreso de admitidos"),
        ("INGRESO_INSCRITOS", "Ingreso relativo")
    ]:

        INDICADORES["B"][
            f"{titulo} promedio S{s}"
        ] = f"PROP_{sufijo}_PROM_S{s}"

        INDICADORES["C"][
            f"{titulo} promedio S{s}"
        ] = f"PROP_{sufijo}_PROM_S{s}"

        INDICADORES["C"][
            f"{titulo} S{s} — 2025"
        ] = f"PROP_{sufijo}_2025_S{s}"

# ============================================================
# 5. DISEÑO INSTITUCIONAL UPB BUCARAMANGA
# ============================================================

imagenes = [
    BASE / "fondo_universidad.jpg",
    BASE / "fondo_universidad.jpeg",
    BASE / "fondo_universidad.png"
]

imagen = next(
    (p for p in imagenes if p.is_file()),
    None
)

if imagen:

    tipo = (
        "png" if imagen.suffix.lower() == ".png"
        else "jpeg"
    )

    imagen_b64 = base64.b64encode(
        imagen.read_bytes()
    ).decode("ascii")

    fondo = (
        "background-image:"
        "linear-gradient("
        "rgba(255,255,255,.87),"
        "rgba(255,255,255,.92)"
        "),"
        f"url('data:image/{tipo};base64,{imagen_b64}');"
        "background-size:cover;"
        "background-position:center;"
        "background-attachment:fixed;"
    )

else:
    fondo = "background-color:#f4f7fb;"

st.markdown(
    """
    <style>

    .stApp {
    """ + fondo + """
    }

    section[data-testid="stSidebar"] {
        background-color:#103b70;
    }

    section[data-testid="stSidebar"] h1,
    section[data-testid="stSidebar"] h2,
    section[data-testid="stSidebar"] h3,
    section[data-testid="stSidebar"] label {
        color:white !important;
    }

    section[data-testid="stSidebar"] [data-baseweb="select"] * {
        color:#172b46;
    }

    div[data-testid="stMetric"] {
        background:rgba(255,255,255,.96);
        border:1px solid #d7e1ec;
        border-radius:12px;
        padding:14px;
    }

    div[data-testid="stMetric"] *,
    section[data-testid="stSidebar"] [data-testid="stMetric"] * {
        color:#172b46 !important;
        opacity:1 !important;
    }

    div[data-testid="stDataFrame"],
    div[data-testid="stPlotlyChart"] {
        background:rgba(255,255,255,.95);
        border-radius:10px;
    }

    h1, h2, h3 {
        color:#143b6c;
    }

    </style>
    """,
    unsafe_allow_html=True
)

# ============================================================
# 6. CARGA Y VALIDACIÓN DE RESULTADOS
# ============================================================

@st.cache_data(show_spinner=False)
def cargar(escenario):

    config = ESCENARIOS[escenario]

    archivo = (
        BASE / f"resultados_cluster_{escenario}.csv"
    )

    tabla = pd.read_csv(
        archivo,
        encoding="utf-8-sig",
        dtype={c: str for c in CLAVES},
        low_memory=False
    )

    dimensiones = (
        config["n"], config["columnas"]
    )

    if tabla.shape != dimensiones:
        raise ValueError(
            f"Escenario {escenario}: dimensiones "
            f"{tabla.shape}; esperadas {dimensiones}"
        )

    requeridas = set(
        CLAVES
        + [
            PROGRAMA,
            IES,
            MUNICIPIO,
            "MODALIDAD",
            "CLUSTER",
            "ESCENARIO"
        ]
        + DUMMIES
        + list(INDICADORES[escenario].values())
    )

    faltantes = requeridas.difference(tabla.columns)

    if faltantes:
        raise ValueError(
            f"Escenario {escenario}: "
            f"faltan columnas {sorted(faltantes)}"
        )

    if tabla.duplicated(CLAVES).any():
        raise ValueError(
            f"Escenario {escenario}: ofertas duplicadas"
        )

    if not tabla["ESCENARIO"].eq(escenario).all():
        raise ValueError(
            f"Escenario {escenario}: "
            "identificación incorrecta"
        )

    tabla["CLUSTER"] = pd.to_numeric(
        tabla["CLUSTER"],
        errors="raise"
    ).astype(int)

    if set(tabla["CLUSTER"]) != set(
        PERFILES[escenario]
    ):
        raise ValueError(
            f"Escenario {escenario}: "
            "clústeres inesperados"
        )

    for columna in set(
        INDICADORES[escenario].values()
    ):

        tabla[columna] = pd.to_numeric(
            tabla[columna],
            errors="raise"
        )

        if tabla[columna].isna().any():
            raise ValueError(
                f"Escenario {escenario}: "
                f"faltantes en {columna}"
            )

    for columna in DUMMIES:

        tabla[columna] = pd.to_numeric(
            tabla[columna],
            errors="raise"
        )

        if not tabla[columna].isin([0, 1]).all():
            raise ValueError(
                f"Escenario {escenario}: "
                f"variable dummy inválida {columna}"
            )

    tabla["PERFIL"] = tabla["CLUSTER"].map(
        PERFILES[escenario]
    )

    return tabla


try:

    tablas = {
        e: cargar(e)
        for e in ESCENARIOS
    }

except Exception as error:

    st.error(
        f"No fue posible cargar los resultados: {error}"
    )

    st.stop()

# ============================================================
# 7. ENCABEZADO Y NAVEGACIÓN
# ============================================================

st.title("🎓 SNIES Analytics")

st.subheader(
    "Segmentación de la demanda académica 2020–2025"
)

st.caption(
    "Minería de Datos · K-means · Metodología CRISP-DM"
)

with st.sidebar:

    st.header("Panel de navegación")

    escenario = st.selectbox(
        "Escenario",
        list(ESCENARIOS),
        key="escenario",
        format_func=lambda e: (
            f"{e} · {ESCENARIOS[e]['nombre']}"
        )
    )

    st.metric(
        "Ofertas del escenario",
        ESCENARIOS[escenario]["n"]
    )

    st.metric(
        "Coeficiente de silueta",
        f"{ESCENARIOS[escenario]['silueta']:.4f}"
    )

    st.info(
        "Análisis descriptivo. "
        "No predice matrícula futura."
    )

# ============================================================
# 8. FILTROS
# ============================================================

original = tablas[escenario]

st.markdown("### Filtros de consulta")

c1, c2, c3 = st.columns(3)

with c1:

    instituciones = st.multiselect(
        "Institución",
        sorted(original[IES].dropna().unique()),
        key=f"institucion_{escenario}"
    )

with c2:

    municipios = st.multiselect(
        "Municipio",
        sorted(original[MUNICIPIO].dropna().unique()),
        key=f"municipio_{escenario}"
    )

with c3:

    programas = st.multiselect(
        "Programa académico",
        sorted(original[PROGRAMA].dropna().unique()),
        key=f"programa_{escenario}"
    )

clusters = st.multiselect(
    "Clúster",
    sorted(PERFILES[escenario]),
    key=f"cluster_{escenario}",
    format_func=lambda k: (
        f"{k} · {PERFILES[escenario][k]}"
    )
)

datos = original.copy()

for campo, seleccion in [
    (IES, instituciones),
    (MUNICIPIO, municipios),
    (PROGRAMA, programas),
    ("CLUSTER", clusters)
]:

    if seleccion:
        datos = datos.loc[
            datos[campo].isin(seleccion)
        ]

# ============================================================
# 9. INDICADORES GENERALES
# ============================================================

a, b, c, d = st.columns(4)

a.metric(
    "Ofertas seleccionadas",
    len(datos)
)

b.metric(
    "Clústeres",
    datos["CLUSTER"].nunique()
)

c.metric(
    "Instituciones",
    datos[IES].nunique()
)

d.metric(
    "Municipios",
    datos[MUNICIPIO].nunique()
)

st.divider()

# ============================================================
# 10. PESTAÑAS
# ============================================================

tab1, tab2, tab3, tab4 = st.tabs([
    "Distribución de clústeres",
    "Consulta de ofertas",
    "Indicadores académicos",
    "Metodología"
])

# ============================================================
# PESTAÑA 1. DISTRIBUCIÓN E INTERPRETACIÓN
# ============================================================

with tab1:

    st.header(
        f"Escenario {escenario} · "
        f"{ESCENARIOS[escenario]['nombre']}"
    )

    if datos.empty:

        st.warning(
            "No hay ofertas para los filtros seleccionados."
        )

    else:

        distribucion = (
            datos.groupby(["CLUSTER", "PERFIL"])
            .size()
            .reset_index(name="OFERTAS")
            .sort_values("CLUSTER")
        )

        distribucion["GRUPO"] = (
            "Clúster "
            + distribucion["CLUSTER"].astype(str)
        )

        fig = px.bar(
            distribucion,
            x="GRUPO",
            y="OFERTAS",
            color="PERFIL",
            text="OFERTAS",
            title="Ofertas académicas por clúster"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.dataframe(
            distribucion[
                ["CLUSTER", "PERFIL", "OFERTAS"]
            ],
            hide_index=True,
            use_container_width=True
        )

        st.caption(
            "Las cantidades reflejan los filtros activos."
        )

    # --------------------------------------------------------
    # AYUDA | INTERPRETACIÓN DE LOS PERFILES ACADÉMICOS
    # --------------------------------------------------------

    with st.expander(
        "📘 Ayuda | ¿Cómo interpretar los perfiles académicos?",
        expanded=False
    ):

        st.markdown(
            "Cada perfil combina características de la demanda "
            "académica y, según el escenario, del ingreso relativo "
            "o de su evolución histórica. Las denominaciones "
            "son comparativas entre los clústeres de un mismo "
            "escenario y no representan categorías ni umbrales "
            "oficiales establecidos por el SNIES."
        )

        st.markdown("#### Conceptos para interpretar los perfiles")

        st.markdown(
            "**Demanda académica:** corresponde al volumen de "
            "personas inscritas en las ofertas académicas. "
            "Los términos *muy baja, baja, intermedia y alta* "
            "expresan diferencias relativas entre los grupos "
            "del escenario analizado."
        )

        st.markdown(
            "**Ingreso relativo:** corresponde al cociente entre "
            "el número de matriculados de primer curso y el "
            "número de inscritos. Un ingreso relativo alto "
            "significa que esta relación es comparativamente "
            "elevada, aunque el volumen total de inscritos "
            "pueda ser reducido."
        )

        st.markdown(
            "**Crecimiento histórico:** describe una pendiente "
            "positiva en la evolución de un indicador durante "
            "el periodo analizado. La pendiente representa "
            "el cambio estimado en estudiantes por año."
        )

        st.markdown(
            "**Estabilidad histórica relativa:** describe "
            "pendientes promedio de pequeña magnitud, cercanas "
            "a cero. No implica necesariamente ausencia de "
            "fluctuaciones ni estabilidad estadística."
        )

        st.divider()

        st.markdown("#### Significado de los perfiles identificados")

        for numero, descripcion in DESCRIPCIONES[escenario].items():

            st.markdown(
                f"**Clúster {numero} — "
                f"{PERFILES[escenario][numero]}**"
            )

            st.write(descripcion)

        st.divider()

        st.caption(
            "Nota metodológica: las interpretaciones describen "
            "características promedio de los grupos obtenidos "
            "mediante K-means. No representan evaluaciones "
            "de calidad académica, relaciones causales ni "
            "predicciones. Los cocientes no deben interpretarse "
            "como probabilidades individuales."
        )

# ============================================================
# PESTAÑA 2. CONSULTA DE OFERTAS
# ============================================================

with tab2:

    st.header("Ofertas académicas clasificadas")

    columnas = [
        PROGRAMA,
        IES,
        MUNICIPIO,
        "MODALIDAD",
        "CLUSTER",
        "PERFIL"
    ] + CLAVES

    st.dataframe(
        datos[columnas],
        hide_index=True,
        use_container_width=True
    )

    st.download_button(
        "Descargar resultados filtrados",
        data=datos.to_csv(
            index=False
        ).encode("utf-8-sig"),
        file_name=f"SNIES_escenario_{escenario}.csv",
        mime="text/csv"
    )

    if not datos.empty:

        consulta = datos.reset_index(drop=True)

        elegido = st.selectbox(
            "Consultar una oferta académica",
            range(len(consulta)),
            key=f"oferta_{escenario}",
            format_func=lambda i: (
                f"{consulta.at[i, PROGRAMA]} | "
                f"{consulta.at[i, IES]} | "
                f"{consulta.at[i, MUNICIPIO]} | "
                f"SNIES {consulta.at[i, CLAVES[0]]} | "
                f"IES {consulta.at[i, CLAVES[1]]} | "
                f"Municipio {consulta.at[i, CLAVES[2]]} | "
                f"Modalidad {consulta.at[i, CLAVES[3]]}"
            )
        )

        oferta = consulta.iloc[elegido]

        st.success(
            f"Clúster {oferta['CLUSTER']}: "
            f"{oferta['PERFIL']}"
        )

        st.write("**Programa:**", oferta[PROGRAMA])
        st.write("**Institución:**", oferta[IES])
        st.write("**Municipio:**", oferta[MUNICIPIO])
        st.write("**Modalidad:**", oferta["MODALIDAD"])
        st.write("**Código SNIES:**", oferta[CLAVES[0]])

# ============================================================
# PESTAÑA 3. INDICADORES ACADÉMICOS
# ============================================================

with tab3:

    st.header("Indicadores académicos")

    if datos.empty:

        st.warning("No hay ofertas para analizar.")

    else:

        opciones = INDICADORES[escenario]

        titulo = st.selectbox(
            "Indicador",
            list(opciones),
            key=f"indicador_{escenario}"
        )

        columna = opciones[titulo]

        grafico = datos.copy()

        grafico["GRUPO"] = (
            "Clúster "
            + grafico["CLUSTER"].astype(str)
        )

        fig = px.box(
            grafico,
            x="GRUPO",
            y=columna,
            color="GRUPO",
            points="outliers",
            title=titulo,
            hover_data=[PROGRAMA, IES]
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        if "Pendiente" in titulo:

            st.caption(
                "Variación estimada en estudiantes/año; "
                "no es una tasa porcentual."
            )

        if "Ingreso relativo" in titulo:

            st.caption(
                "Cociente entre matriculados de primer "
                "curso e inscritos; no es una "
                "probabilidad individual."
            )

        # ----------------------------------------------------
        # COMPARACIÓN HISTÓRICA DEL ESCENARIO C
        # ----------------------------------------------------

        if escenario == "C":

            st.markdown(
                "### Comparación histórica frente a 2025"
            )

            variable = st.radio(
                "Variable",
                [
                    "INSCRITOS",
                    "MATRICULADOS PRIMER CURSO"
                ],
                horizontal=True,
                key="variable_comparacion_C"
            )

            semestre = st.radio(
                "Semestre",
                [1, 2],
                horizontal=True,
                key="semestre_comparacion_C"
            )

            historica = (
                f"{variable}_PROM_S{semestre}"
            )

            reciente = (
                f"{variable}_2025_S{semestre}"
            )

            comparacion = (
                datos.groupby("CLUSTER")[
                    [historica, reciente]
                ]
                .mean()
                .reset_index()
                .melt(
                    id_vars="CLUSTER",
                    var_name="PERIODO",
                    value_name="PROMEDIO"
                )
            )

            comparacion["PERIODO"] = (
                comparacion["PERIODO"].replace({
                    historica: "Promedio histórico",
                    reciente: "Valor de 2025"
                })
            )

            comparacion["GRUPO"] = (
                "Clúster "
                + comparacion["CLUSTER"].astype(str)
            )

            fig_c = px.bar(
                comparacion,
                x="GRUPO",
                y="PROMEDIO",
                color="PERIODO",
                barmode="group",
                text_auto=".1f",
                title=(
                    f"{variable} · Semestre {semestre}: "
                    "histórico frente a 2025"
                )
            )

            st.plotly_chart(
                fig_c,
                use_container_width=True
            )

            st.caption(
                "Comparación descriptiva del promedio "
                "histórico con el registro de 2025."
            )

# ============================================================
# PESTAÑA 4. METODOLOGÍA
# ============================================================

with tab4:

    st.header("Metodología CRISP-DM")

    st.markdown("""
**Objetivo:** caracterizar la demanda académica mediante
agrupamiento no supervisado K-means.

**Unidad de análisis:** oferta académica identificada
mediante código SNIES, institución, municipio y modalidad.

**Escenario A:** comportamiento de los dos semestres
de 2025 (124 ofertas, 8 clústeres).

**Escenario B:** evolución de 2020 a 2025, con un mínimo
de cuatro años completos (115 ofertas, 5 clústeres).

**Escenario C:** integración de características históricas
y recientes (105 ofertas, 2 clústeres).

**Preparación:** selección de registros Masculino y Femenino
para construir los indicadores académicos; normalización
MinMax de variables cuantitativas y codificación dummy
del sector y tipo de institución.

La modalidad se conserva como variable contextual
y no interviene en el cálculo de distancias de K-means.

**Centroides:** se recuperan de `cluster_centers_` y no
se recalculan durante el perfilamiento. Las variables
cuantitativas se desnormalizan mediante `inverse_transform()`.

Las variables `SECTOR IES_Privado` y `TIPO IES_Seccional`
expresan la proporción de ofertas de esas categorías
dentro del clúster y no requieren desnormalización.

**Evaluación:** coeficiente de silueta, distribución de
ofertas y caracterización académica de los centroides.

**Limitaciones:** los centroides representan promedios
y no describen necesariamente cada oferta individual.
Los cocientes pueden superar 1 y no deben interpretarse
como probabilidades individuales. Las pendientes representan
cambios absolutos estimados en estudiantes/año, no pruebas
de significancia estadística.

Los escenarios utilizan poblaciones y características
distintas; sus coeficientes de silueta no constituyen
una comparación directa de desempeño.

Los resultados son exploratorios, no predictivos ni causales.
    """)

    resumen = pd.DataFrame([
        {
            "Escenario": e,
            "Ofertas": config["n"],
            "Clústeres": config["k"],
            "Silueta": config["silueta"]
        }
        for e, config in ESCENARIOS.items()
    ])

    st.dataframe(
        resumen,
        hide_index=True,
        use_container_width=True
    )

st.divider()

st.caption(
    "Proyecto de Minería de Datos · "
    "SNIES 2020–2025 · Streamlit"
)
'''

# ============================================================
# 2. VALIDACIÓN ANTES DE MODIFICAR app.py
# ============================================================

ast.parse(codigo_app)

assert codigo_app.count("DESCRIPCIONES = {") == 1

assert codigo_app.count(
    "Ayuda | ¿Cómo interpretar los perfiles académicos?"
) == 1

esperado = {
    "A": (124, 24, 8),
    "B": (115, 28, 5),
    "C": (105, 38, 2)
}

for escenario, (n, columnas, k) in esperado.items():

    ruta = (
        carpeta / f"resultados_cluster_{escenario}.csv"
    )

    assert ruta.is_file(), (
        f"Falta el archivo {ruta.name}"
    )

    tabla = pd.read_csv(
        ruta,
        encoding="utf-8-sig"
    )

    assert tabla.shape == (n, columnas), (
        f"Escenario {escenario}: "
        f"dimensiones incorrectas {tabla.shape}"
    )

    assert tabla["CLUSTER"].nunique() == k, (
        f"Escenario {escenario}: "
        "número de clústeres incorrecto"
    )

ruta_consolidado = (
    carpeta / "resultados_cluster_consolidados.csv"
)

assert ruta_consolidado.is_file()

assert len(
    pd.read_csv(ruta_consolidado)
) == 344

# ============================================================
# 3. RESPALDO AUTOMÁTICO
# ============================================================

ruta_app = carpeta / "app.py"
ruta_requirements = carpeta / "requirements.txt"

if ruta_app.exists():

    respaldo = carpeta / (
        f"app_respaldo_"
        f"{datetime.now():%Y%m%d_%H%M%S_%f}.py"
    )

    respaldo.write_bytes(
        ruta_app.read_bytes()
    )

    print("Respaldo creado:", respaldo)

# ============================================================
# 4. GUARDAR app.py Y requirements.txt
# ============================================================

ruta_app.write_text(
    codigo_app.strip() + "\n",
    encoding="utf-8"
)

ruta_requirements.write_text(
    "streamlit>=1.40,<2\n"
    "pandas>=2.2,<3\n"
    "plotly>=5.20,<7\n",
    encoding="utf-8"
)

# ============================================================
# 5. REPORTE FINAL
# ============================================================

print("\n" + "=" * 65)
print("SNIES ANALYTICS | CELDA 2 ACTUALIZADA")
print("=" * 65)

print("Sintaxis de app.py: CORRECTA")
print("Escenario A: 8 perfiles y descripciones")
print("Escenario B: 5 perfiles y descripciones")
print("Escenario C: 2 perfiles y descripciones")
print("Total: 15 clústeres académicos")
print("Ayuda e interpretación: INCORPORADA")
print("Pestañas Streamlit: 4")
print("Archivos CSV: SIN MODIFICACIONES")
print("Modelos K-means: SIN MODIFICACIONES")

print("\nAplicación:", ruta_app)
print("Dependencias:", ruta_requirements)
print("\nCELDA 2 FINALIZADA CORRECTAMENTE")


In [ ]:
import sys
import subprocess
from pathlib import Path

carpeta = Path(
    "/content/drive/MyDrive/"
    "MaestriaUPB_Mineria_Datos/P_final/streamlit"
)

requirements = carpeta / "requirements.txt"

assert requirements.is_file(), "No se encontró requirements.txt"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements)
    ],
    check=True
)

print("Dependencias instaladas correctamente.")

In [ ]:

# ============================================================
# SNIES ANALYTICS 2020-2025
# CELDA 4 | INICIO Y PRUEBA FUNCIONAL DE STREAMLIT
# GOOGLE COLAB
#
# - Verifica app.py y los CSV definitivos.
# - Comprueba las 15 descripciones académicas.
# - Reinicia únicamente Streamlit de este proyecto.
# - Aplica compatibilidad WebSocket para Google Colab.
# - Espera a que el servidor responda.
# - Genera un enlace de acceso de la sesión actual.
#
# No modifica app.py, CSV, matrices ni modelos K-means.
# Configuración de seguridad reducida SOLO para Colab.
# ============================================================

import sys
import os
import ast
import signal
import subprocess
import time
import urllib.request
from pathlib import Path
from html import escape

from google.colab import output
from IPython.display import display, HTML

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

carpeta = Path(
    "/content/drive/MyDrive/"
    "MaestriaUPB_Mineria_Datos/P_final/streamlit"
)

app = carpeta / "app.py"
log = Path("/content/streamlit_snies.log")

puerto = 8501

archivos = [
    "app.py",
    "requirements.txt",
    "resultados_cluster_A.csv",
    "resultados_cluster_B.csv",
    "resultados_cluster_C.csv",
    "resultados_cluster_consolidados.csv"
]

# ============================================================
# 2. VALIDAR ARCHIVOS Y APLICACIÓN
# ============================================================

for nombre in archivos:
    assert (carpeta / nombre).is_file(), (
        f"Archivo no encontrado: {nombre}"
    )

codigo = app.read_text(encoding="utf-8")

ast.parse(codigo)

assert "DESCRIPCIONES = {" in codigo, (
    "app.py no contiene las descripciones académicas."
)

assert "DESCRIPCIONES = {" in codigo, (
    "app.py no contiene las descripciones académicas."
)

assert "Ayuda | ¿Cómo interpretar los perfiles académicos?" in codigo, (
    "No se encontró el título actualizado de la sección de ayuda."
)

assert "Conceptos para interpretar los perfiles" in codigo, (
    "No se encontró el glosario de conceptos académicos."
)

assert "Significado de los perfiles identificados" in codigo, (
    "No se encontró la interpretación de los 15 perfiles."
)

imagenes = [
    carpeta / "fondo_universidad.jpg",
    carpeta / "fondo_universidad.jpeg",
    carpeta / "fondo_universidad.png"
]

imagen = next(
    (p for p in imagenes if p.is_file()),
    None
)

print("=" * 65, flush=True)
print("SNIES ANALYTICS | PRUEBA FUNCIONAL", flush=True)
print("=" * 65, flush=True)

print("app.py: CORRECTO", flush=True)
print("Archivos CSV: DISPONIBLES", flush=True)
print("Sintaxis Python: CORRECTA", flush=True)
print("Descripciones de clústeres: INCORPORADAS", flush=True)

print(
    "Fotografía institucional:",
    imagen.name if imagen else "No disponible",
    flush=True
)

# ============================================================
# 3. FUNCIÓN DE VERIFICACIÓN HTTP
# ============================================================

url_salud = (
    f"http://127.0.0.1:{puerto}/_stcore/health"
)

def comprobar_servidor():
    try:
        with urllib.request.urlopen(
            url_salud,
            timeout=1
        ) as respuesta:
            return respuesta.status == 200
    except Exception:
        return False

# ============================================================
# 4. CERRAR SOLO STREAMLIT DE SNIES ANALYTICS
# ============================================================

procesos = subprocess.run(
    ["ps", "-eo", "pid,args"],
    capture_output=True,
    text=True,
    timeout=5,
    check=True
)

pids = []

for linea in procesos.stdout.splitlines()[1:]:

    partes = linea.strip().split(maxsplit=1)

    if len(partes) != 2:
        continue

    pid_texto, comando_actual = partes

    if (
        "-m streamlit run" in comando_actual
        and str(app) in comando_actual
    ):
        pid = int(pid_texto)

        if pid != os.getpid():
            pids.append(pid)

for pid in pids:

    try:
        os.kill(pid, signal.SIGTERM)

        print(
            f"Servidor anterior detenido: PID {pid}",
            flush=True
        )

    except ProcessLookupError:
        pass

# Espera controlada para liberar el puerto
for intento in range(10):

    if not comprobar_servidor():
        break

    time.sleep(0.5)

if comprobar_servidor():
    raise RuntimeError(
        "El puerto 8501 continúa ocupado. "
        "No se iniciará otra instancia."
    )

# ============================================================
# 5. INICIAR STREAMLIT PARA GOOGLE COLAB
# ============================================================

print("\nIniciando Streamlit...", flush=True)

comando = [
    sys.executable,
    "-m", "streamlit", "run",
    str(app),
    f"--server.port={puerto}",
    "--server.address=0.0.0.0",
    "--server.headless=true",

    # Compatibilidad con el proxy temporal de Colab
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",

    "--browser.gatherUsageStats=false"
]

with log.open("w", encoding="utf-8") as archivo_log:

    proceso = subprocess.Popen(
        comando,
        cwd=str(carpeta),
        stdout=archivo_log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print(
    "Proceso Streamlit iniciado: PID",
    proceso.pid,
    flush=True
)

# ============================================================
# 6. ESPERA CONTROLADA DEL SERVIDOR
# ============================================================

activo = False

for intento in range(30):

    if proceso.poll() is not None:
        break

    if comprobar_servidor():
        activo = True
        break

    time.sleep(1)

# ============================================================
# 7. RESULTADO DEL SERVIDOR
# ============================================================

print("\n" + "=" * 65, flush=True)
print("SNIES ANALYTICS | RESULTADO", flush=True)
print("=" * 65, flush=True)

if not activo:

    print("Servidor Streamlit: NO DISPONIBLE")

    if log.exists():

        registro = log.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print("\nÚltimos mensajes del servidor:")
        print(
            registro[-4000:]
            if registro.strip()
            else "Registro vacío."
        )

    raise RuntimeError(
        "No fue posible verificar el inicio de Streamlit."
    )

print("Servidor Streamlit: ACTIVO", flush=True)
print("Respuesta HTTP: 200", flush=True)
print("Puerto:", puerto, flush=True)
print("Compatibilidad WebSocket Colab: ACTIVADA", flush=True)

# ============================================================
# 8. ENLACE DEL PROXY DE LA SESIÓN ACTUAL
# ============================================================

print("\nGenerando enlace de acceso...", flush=True)

try:

    enlace = output.eval_js(
        f"google.colab.kernel.proxyPort({puerto})"
    )

    if not (
        isinstance(enlace, str)
        and enlace.startswith("https://")
    ):
        raise ValueError(
            "Colab no devolvió una URL HTTPS válida."
        )

    print("\nENLACE SNIES ANALYTICS:")
    print(enlace, flush=True)

    enlace_seguro = escape(enlace, quote=True)

    display(HTML(
        f'<a href="{enlace_seguro}" '
        'target="_blank" '
        'rel="noopener noreferrer" '
        'style="font-size:18px;font-weight:bold;">'
        '🎓 ABRIR SNIES ANALYTICS'
        '</a>'
    ))

except Exception as error:

    print(
        "\nEl servidor funciona, pero Colab no pudo "
        "generar automáticamente el enlace."
    )

    print("Detalle:", error)

# ============================================================
# 9. VERIFICACIÓN FINAL
# ============================================================

print("\n" + "=" * 65)
print("SNIES ANALYTICS | PRUEBA PREPARADA")
print("=" * 65)

print("Modelos K-means: SIN MODIFICACIONES")
print("CSV: SIN MODIFICACIONES")
print("app.py: SIN MODIFICACIONES")
print("15 descripciones: DISPONIBLES")
print("Servidor: ACTIVO")

print(
    "\nAbre el enlace y verifica las cuatro pestañas "
    "de SNIES Analytics."
)

print(
    "\nNOTA: Esta configuración de CORS y XSRF "
    "es exclusiva para pruebas temporales en Colab."
)


#GitHub

In [ ]:
# ============================================================
# SNIES ANALYTICS | PREPARACIÓN PARA GITHUB
# PASO 1: INVENTARIO DE ARCHIVOS
# ============================================================

from pathlib import Path

base = Path(
    "/content/drive/MyDrive/"
    "MaestriaUPB_Mineria_Datos/P_final"
)

streamlit_dir = base / "streamlit"

archivos = [
    "app.py",
    "requirements.txt",
    "fondo_universidad.jpg",
    "resultados_cluster_A.csv",
    "resultados_cluster_B.csv",
    "resultados_cluster_C.csv",
    "resultados_cluster_consolidados.csv"
]

print("=" * 65)
print("SNIES ANALYTICS | INVENTARIO PARA GITHUB")
print("=" * 65)

total = 0
faltantes = []

for nombre in archivos:

    ruta = streamlit_dir / nombre

    if ruta.is_file():

        tamano = ruta.stat().st_size / (1024 ** 2)
        total += ruta.stat().st_size

        print(f"OK  | {nombre} | {tamano:.2f} MB")

    else:

        faltantes.append(nombre)
        print(f"FALTA | {nombre}")

print("\nNOTEBOOKS DISPONIBLES:")

notebooks = sorted(base.rglob("*.ipynb"))

if notebooks:

    for notebook in notebooks:

        print(
            notebook.relative_to(base),
            f"| {notebook.stat().st_size / (1024 ** 2):.2f} MB"
        )

else:

    print("No se encontraron notebooks .ipynb en P_final.")

print("\n" + "=" * 65)
print("RESULTADO")
print("=" * 65)

print("Archivos de Streamlit encontrados:",
      len(archivos) - len(faltantes), "/", len(archivos))

print("Tamaño total:",
      round(total / (1024 ** 2), 2), "MB")

print("Archivos faltantes:", faltantes or "Ninguno")

print("\nNo se modificó ningún archivo.")

In [ ]:

# ============================================================
# SNIES ANALYTICS 2020-2025
# PASO 2 | PREPARAR ENTREGA ACADÉMICA PARA GITHUB
#
# - Copia los 7 archivos necesarios de Streamlit.
# - Prepara el notebook sin salidas de ejecución.
# - Genera README.md con la información académica.
# - Comprueba archivos, sintaxis y posibles credenciales.
# - No modifica los originales de Google Drive.
# - No publica todavía en GitHub.
# ============================================================

from pathlib import Path
import shutil
import json
import re
import ast
import hashlib
import textwrap

# ============================================================
# 1. RUTAS
# ============================================================

origen = Path(
    "/content/drive/MyDrive/"
    "MaestriaUPB_Mineria_Datos/P_final"
)

origen_streamlit = origen / "streamlit"

destino = Path(
    "/content/SNIES-Analytics-2020-2025"
)

destino_streamlit = destino / "streamlit"
destino_notebooks = destino / "notebooks"

destino_streamlit.mkdir(
    parents=True,
    exist_ok=True
)

destino_notebooks.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 65)
print("SNIES ANALYTICS | PREPARACIÓN PARA GITHUB")
print("=" * 65)

# ============================================================
# 2. IDENTIFICAR Y VALIDAR LOS ARCHIVOS
# ============================================================

archivos_streamlit = [
    "app.py",
    "requirements.txt",
    "fondo_universidad.jpg",
    "resultados_cluster_A.csv",
    "resultados_cluster_B.csv",
    "resultados_cluster_C.csv",
    "resultados_cluster_consolidados.csv"
]

notebook_origen = (
    origen / "Clustering_T_F_Maryory.ipynb"
)

for nombre in archivos_streamlit:
    ruta = origen_streamlit / nombre

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró: {ruta}"
        )

if not notebook_origen.is_file():
    raise FileNotFoundError(
        f"No se encontró: {notebook_origen}"
    )

# Validar sintaxis de la aplicación
codigo_app = (
    origen_streamlit / "app.py"
).read_text(encoding="utf-8")

ast.parse(codigo_app)

# ============================================================
# 3. REVISIÓN PREVENTIVA DE CREDENCIALES
# ============================================================

notebook = json.loads(
    notebook_origen.read_text(
        encoding="utf-8"
    )
)

celdas = notebook.get("cells", [])

codigo_notebook = "\n".join(
    "".join(celda.get("source", []))
    for celda in celdas
    if celda.get("cell_type") == "code"
)

patrones = {
    "Token clásico de GitHub":
        r"gh[pousr]_[A-Za-z0-9_]{20,}",

    "Token de GitHub fine-grained":
        r"github_pat_[A-Za-z0-9_]{20,}",

    "Clave privada":
        r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",

    "Asignación literal de credencial":
        r"""(?i)(?:api_key|access_token|secret_key|password)\s*=\s*['"][^'"]{8,}['"]"""
}

fuentes = {
    "app.py": codigo_app,
    "notebook": codigo_notebook
}

alertas = []

for archivo, texto_fuente in fuentes.items():

    for nombre, patron in patrones.items():

        if re.search(patron, texto_fuente):
            alertas.append(
                f"{archivo}: {nombre}"
            )

print("\nREVISIÓN DE SEGURIDAD")

if alertas:

    print("Se encontraron posibles credenciales:")

    for alerta in alertas:
        print("ALERTA:", alerta)

    raise RuntimeError(
        "Revisar las posibles credenciales "
        "antes de preparar la publicación."
    )

else:
    print("No se detectaron credenciales con los patrones revisados.")

# ============================================================
# 4. COPIAR LOS ARCHIVOS DE STREAMLIT
# ============================================================

print("\nCOPIANDO ARCHIVOS DE STREAMLIT")

for nombre in archivos_streamlit:

    fuente = origen_streamlit / nombre
    objetivo = destino_streamlit / nombre

    shutil.copy2(
        fuente,
        objetivo
    )

    print("OK:", nombre)

# ============================================================
# 5. PREPARAR NOTEBOOK SIN SALIDAS
# ============================================================

salidas_eliminadas = 0

for celda in celdas:

    if celda.get("cell_type") == "code":

        if celda.get("outputs"):
            salidas_eliminadas += 1

        celda["outputs"] = []
        celda["execution_count"] = None

notebook_destino = (
    destino_notebooks /
    "Clustering_T_F_Maryory.ipynb"
)

notebook_destino.write_text(
    json.dumps(
        notebook,
        ensure_ascii=False,
        indent=1
    ),
    encoding="utf-8"
)

print("\nNOTEBOOK")
print("Archivo:", notebook_destino.name)
print(
    "Celdas con salidas eliminadas:",
    salidas_eliminadas
)

# ============================================================
# 6. GENERAR README.md
#
# Se utilizan comillas triples simples para evitar
# el error de cierre del bloque de texto anterior.
# ============================================================

readme = textwrap.dedent('''
# SNIES Analytics 2020–2025

## Información académica

**Universidad:** Universidad Pontificia Bolivariana (UPB)
**Programa:** Maestría en Ciencia de Datos
**Asignatura:** Minería de Datos
**Docente:** Ana Oviedo
**Elaborado por:** Maryory Patricia Villamizar León
**Año:** 2026

## Descripción del proyecto

SNIES Analytics 2020–2025 es una solución de minería
de datos orientada a caracterizar y segmentar la demanda
académica de programas de educación superior en Colombia.

El proyecto utiliza información del Sistema Nacional
de Información de la Educación Superior (SNIES),
técnicas de preparación de datos y aprendizaje
no supervisado mediante el algoritmo K-means.

Los resultados se presentan mediante una aplicación
interactiva desarrollada con Streamlit.

## Objetivo general

Caracterizar y segmentar la demanda académica de
programas de educación superior en Colombia mediante
técnicas de minería de datos, con el fin de identificar
perfiles de comportamiento reciente e histórico.

## Metodología CRISP-DM

El desarrollo considera las seis etapas:

1. Comprensión del negocio.
2. Comprensión de los datos.
3. Preparación de los datos.
4. Modelado.
5. Evaluación.
6. Despliegue.

## Escenarios de agrupamiento

| Escenario | Enfoque | Ofertas | Clústeres | Silueta |
|---|---|---:|---:|---:|
| A | Demanda reciente de 2025 | 124 | 8 | 0.5300 |
| B | Evolución histórica 2020–2025 | 115 | 5 | 0.4452 |
| C | Demanda integral | 105 | 2 | 0.3973 |

Los escenarios utilizan diferentes conjuntos de ofertas
y características, por lo que sus coeficientes de silueta
no deben utilizarse como una comparación directa
de desempeño entre escenarios.

## Estructura del repositorio

- `README.md`: descripción general del proyecto.
- `notebooks/Clustering_T_F_Maryory.ipynb`:
  desarrollo analítico y metodológico.
- `streamlit/app.py`: aplicación interactiva.
- `streamlit/requirements.txt`: dependencias.
- `streamlit/fondo_universidad.jpg`:
  imagen de fondo de la aplicación.
- `streamlit/resultados_cluster_A.csv`:
  clasificación del escenario A.
- `streamlit/resultados_cluster_B.csv`:
  clasificación del escenario B.
- `streamlit/resultados_cluster_C.csv`:
  clasificación del escenario C.
- `streamlit/resultados_cluster_consolidados.csv`:
  resultados integrados.

## Ejecución de la aplicación

Instalar las dependencias desde la carpeta `streamlit`:

    pip install -r requirements.txt

Iniciar la aplicación:

    streamlit run app.py

La aplicación presenta cuatro secciones:

1. Distribución de clústeres.
2. Consulta de ofertas académicas.
3. Indicadores académicos.
4. Metodología CRISP-DM.

## Interpretación de los resultados

Los clústeres identifican perfiles relativos de demanda
académica, ingreso relativo y evolución histórica.

Las interpretaciones son descriptivas y exploratorias.
No representan predicciones, relaciones causales
ni categorías oficiales establecidas por el SNIES.

## Fuente de datos

Sistema Nacional de Información de la Educación
Superior (SNIES), Colombia.

Período de análisis: 2020–2025.

## Autoría académica

**Maryory Patricia Villamizar León**

Maestría en Ciencia de Datos
Universidad Pontificia Bolivariana – UPB
2026
''').strip() + "\n"

readme_destino = destino / "README.md"

readme_destino.write_text(
    readme,
    encoding="utf-8"
)

print("\nREADME ACADÉMICO")
print("Archivo:", readme_destino.name)
print("Asignatura: Minería de Datos")
print("Docente: Ana Oviedo")
print("Autora: Maryory Patricia Villamizar León")

# ============================================================
# 7. VERIFICAR COPIAS
# ============================================================

def sha256_archivo(ruta):

    digest = hashlib.sha256()

    with ruta.open("rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(bloque)

    return digest.hexdigest()

for nombre in archivos_streamlit:

    original = origen_streamlit / nombre
    copia = destino_streamlit / nombre

    assert sha256_archivo(original) == sha256_archivo(copia), (
        f"La copia de {nombre} no coincide."
    )

ast.parse(
    (destino_streamlit / "app.py").read_text(
        encoding="utf-8"
    )
)

json.loads(
    notebook_destino.read_text(
        encoding="utf-8"
    )
)

assert readme_destino.stat().st_size > 0

# ============================================================
# 8. INFORME FINAL
# ============================================================

print("\n" + "=" * 65)
print("SNIES ANALYTICS | ENTREGA PREPARADA")
print("=" * 65)

total_bytes = 0
total_archivos = 0

for ruta in sorted(destino.rglob("*")):

    if ruta.is_file():

        total_archivos += 1
        total_bytes += ruta.stat().st_size

        print(
            "OK:",
            ruta.relative_to(destino)
        )

print("\nArchivos preparados:", total_archivos)

print(
    "Tamaño total:",
    f"{total_bytes / (1024 ** 2):.2f} MB"
)

print("\nVALIDACIONES:")
print("Sintaxis app.py: CORRECTA")
print("Notebook JSON: CORRECTO")
print("Copias Streamlit: VERIFICADAS")
print("README.md: CREADO")

print("\nCarpeta preparada:", destino)

print("\nProyecto original de Drive: SIN MODIFICACIONES")
print("Repositorio GitHub: TODAVÍA SIN CAMBIOS")

print("\nPASO 2 FINALIZADO")


In [ ]:

# ============================================================
# SNIES ANALYTICS 2020-2025
# PASO 3 | PUBLICACIÓN DIRECTA EN GITHUB
#
# Repositorio:
# Maryory-Villamizar/SNIES-Analytics-2020-2025
#
# - Solicita el token de forma oculta.
# - Publica los 9 archivos preparados.
# - Realiza un único commit en main.
# - Verifica los archivos publicados.
# - No modifica los archivos de Google Drive.
# ============================================================

from pathlib import Path
from getpass import getpass
import urllib.request
import urllib.error
import json
import base64
import hashlib

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

USUARIO = "Maryory-Villamizar"
REPOSITORIO = "SNIES-Analytics-2020-2025"
RAMA = "main"

CARPETA = Path(
    "/content/SNIES-Analytics-2020-2025"
)

API = (
    f"https://api.github.com/repos/"
    f"{USUARIO}/{REPOSITORIO}"
)

ARCHIVOS = [
    "README.md",
    "notebooks/Clustering_T_F_Maryory.ipynb",
    "streamlit/app.py",
    "streamlit/requirements.txt",
    "streamlit/fondo_universidad.jpg",
    "streamlit/resultados_cluster_A.csv",
    "streamlit/resultados_cluster_B.csv",
    "streamlit/resultados_cluster_C.csv",
    "streamlit/resultados_cluster_consolidados.csv"
]

print("=" * 65)
print("SNIES ANALYTICS | PUBLICACIÓN EN GITHUB")
print("=" * 65)

# ============================================================
# 2. VERIFICAR LOS 9 ARCHIVOS
# ============================================================

for nombre in ARCHIVOS:

    ruta = CARPETA / nombre

    if not ruta.is_file():
        raise FileNotFoundError(
            f"Falta el archivo: {nombre}"
        )

print("\nArchivos verificados:", len(ARCHIVOS))

for nombre in ARCHIVOS:
    print("OK:", nombre)

# ============================================================
# 3. SOLICITAR TOKEN DE FORMA OCULTA
# ============================================================

print("\nAUTENTICACIÓN GITHUB")
print(
    "Pega el token cuando aparezca el campo oculto."
)
print(
    "No se mostrará en pantalla ni se guardará "
    "en los archivos del proyecto."
)

TOKEN = getpass(
    "Token personal de GitHub: "
).strip()

if not TOKEN:
    raise ValueError(
        "No se ingresó ningún token."
    )

# ============================================================
# 4. FUNCIÓN PARA COMUNICARSE CON GITHUB
# ============================================================

def github_api(
    endpoint,
    metodo="GET",
    datos=None
):

    direccion = API + endpoint

    encabezados = {
        "Authorization": f"Bearer {TOKEN}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "SNIES-Analytics-Colab"
    }

    cuerpo = None

    if datos is not None:

        cuerpo = json.dumps(
            datos
        ).encode("utf-8")

        encabezados["Content-Type"] = (
            "application/json"
        )

    solicitud = urllib.request.Request(
        direccion,
        data=cuerpo,
        headers=encabezados,
        method=metodo
    )

    try:

        with urllib.request.urlopen(
            solicitud,
            timeout=120
        ) as respuesta:

            contenido = respuesta.read()

            return (
                json.loads(contenido)
                if contenido
                else {}
            )

    except urllib.error.HTTPError as error:

        try:
            detalle = json.loads(
                error.read().decode("utf-8")
            ).get("message", "Sin detalle")
        except Exception:
            detalle = "Sin detalle"

        raise RuntimeError(
            f"GitHub respondió HTTP {error.code}. "
            f"Detalle: {detalle}"
        ) from None

# ============================================================
# 5. VERIFICAR REPOSITORIO Y RAMA
# ============================================================

try:

    informacion = github_api("")

    print("\nREPOSITORIO")
    print("Nombre:", informacion["full_name"])

    if informacion["full_name"].lower() != (
        f"{USUARIO}/{REPOSITORIO}".lower()
    ):
        raise RuntimeError(
            "El repositorio no coincide."
        )

    referencia = github_api(
        f"/git/ref/heads/{RAMA}"
    )

    commit_anterior = referencia[
        "object"
    ]["sha"]

    commit_info = github_api(
        f"/git/commits/{commit_anterior}"
    )

    arbol_anterior = commit_info[
        "tree"
    ]["sha"]

    print("Rama:", RAMA)
    print(
        "Commit anterior:",
        commit_anterior[:12]
    )

    # ========================================================
    # 6. CREAR BLOBS DE LOS ARCHIVOS
    # ========================================================

    print("\nENVIANDO ARCHIVOS A GITHUB")

    elementos_arbol = []
    hashes_esperados = {}

    for indice, nombre in enumerate(
        ARCHIVOS,
        start=1
    ):

        ruta = CARPETA / nombre

        contenido = ruta.read_bytes()

        contenido_b64 = base64.b64encode(
            contenido
        ).decode("ascii")

        resultado = github_api(
            "/git/blobs",
            metodo="POST",
            datos={
                "content": contenido_b64,
                "encoding": "base64"
            }
        )

        sha = resultado["sha"]

        elementos_arbol.append({
            "path": nombre,
            "mode": "100644",
            "type": "blob",
            "sha": sha
        })

        hashes_esperados[nombre] = sha

        print(
            f"[{indice}/{len(ARCHIVOS)}] "
            f"Enviado: {nombre}"
        )

    # ========================================================
    # 7. PREPARAR UNA SOLA ACTUALIZACIÓN
    # ========================================================

    arbol_nuevo = github_api(
        "/git/trees",
        metodo="POST",
        datos={
            "base_tree": arbol_anterior,
            "tree": elementos_arbol
        }
    )

    nuevo_commit = github_api(
        "/git/commits",
        metodo="POST",
        datos={
            "message": (
                "Publicación académica inicial "
                "SNIES Analytics 2020-2025"
            ),
            "tree": arbol_nuevo["sha"],
            "parents": [commit_anterior]
        }
    )

    # ========================================================
    # 8. ACTUALIZAR LA RAMA MAIN
    # ========================================================

    print(
        "\nPUBLICANDO COMMIT EN LA RAMA MAIN..."
    )

    github_api(
        f"/git/refs/heads/{RAMA}",
        metodo="PATCH",
        datos={
            "sha": nuevo_commit["sha"],
            "force": False
        }
    )

    print("Commit publicado correctamente.")

    # ========================================================
    # 9. VERIFICAR ARCHIVOS EN GITHUB
    # ========================================================

    referencia_final = github_api(
        f"/git/ref/heads/{RAMA}"
    )

    if referencia_final["object"]["sha"] != (
        nuevo_commit["sha"]
    ):
        raise RuntimeError(
            "La rama main no apunta al commit esperado."
        )

    arbol_verificacion = github_api(
        f"/git/trees/{arbol_nuevo['sha']}"
        "?recursive=1"
    )

    archivos_remotos = {
        elemento["path"]: elemento["sha"]
        for elemento in arbol_verificacion["tree"]
        if elemento["type"] == "blob"
    }

    faltantes = [
        nombre
        for nombre in ARCHIVOS
        if archivos_remotos.get(nombre)
        != hashes_esperados[nombre]
    ]

    if faltantes:
        raise RuntimeError(
            "No se pudo confirmar la publicación "
            f"de estos archivos: {faltantes}"
        )

    # ========================================================
    # 10. RESULTADO FINAL
    # ========================================================

    enlace = (
        f"https://github.com/"
        f"{USUARIO}/{REPOSITORIO}"
    )

    print("\n" + "=" * 65)
    print("SNIES ANALYTICS | PUBLICACIÓN EXITOSA")
    print("=" * 65)

    print(
        "Archivos verificados en GitHub:",
        len(ARCHIVOS)
    )

    print(
        "Commit:",
        nuevo_commit["sha"]
    )

    print("Rama:", RAMA)

    print("\nREPOSITORIO:")
    print(enlace)

    print(
        "\nProyecto original de Drive: "
        "SIN MODIFICACIONES"
    )

    print("\nENTREGA PUBLICADA CORRECTAMENTE")

finally:

    # Eliminar la variable global del token
    # después de finalizar o fallar la operación.
    if "TOKEN" in globals():
        del TOKEN
